# Vector RAG Pipeline — FinanceBench (full batch run)

**Paper:** Kim et al. (2025), _GAR: Generative Answer Refinement for Financial QA_, arXiv 2503.15191
**Thesis:** Vector RAG vs. Vectorless RAG vs. Long-Context LLMs on FinanceBench

---

## What changed in this version

Two changes from the previous version of this notebook:

1. **Every function now lives in this notebook**, not in `pipelines/vector_rag/*.py`.
   Nothing outside this notebook imported those modules, so they were folded in here
   rather than kept as a second copy. This is deliberately the working structure for
   now — once all three pipelines (vector RAG, vectorless RAG, long-context) run
   end to end, the common pieces (retrieval metrics, answer scoring, cost tracking)
   are worth re-extracting into a shared module again. `evaluation/*.py` still exists
   on disk with the same logic (it's documented in CLAUDE.md as shared across all
   three pipelines) — this notebook just doesn't import it anymore, to keep
   everything walkable top-to-bottom in one place.

2. **What used to be one "Stage 5" cell (retrieve → rerank → select → generate →
   score) is now five separate stages**, each saving its own output file for all
   150 questions before the next stage starts — the same resumable, append-and-skip
   pattern Stage 4/5 below already use for indexing and query embedding. This means
   you can inspect retrieval quality, or read what got selected, or read the raw
   generated answers, independently, without needing the whole pipeline to have
   finished.

   **Trade-off:** this makes per-question *end-to-end latency* (query submission to
   answer) unmeasurable from these stages, since two stages for the same question
   can now run hours or days apart across separate resumable runs. The project
   brief wants a median-of-N latency number — that still needs a dedicated
   sequential timing pass (all steps back-to-back, no resuming) over some/all
   questions, done separately and documented as such in the thesis methodology.
   The split stages below do NOT produce a latency number for this reason.

```
Stage 0   Setup             — Drive mount, repo clone/pull, API keys
Stage 1   Shared utilities  — retry/backoff, cost tracker, model clients
Stage 2   Load data         — 150 questions, 84 unique documents
Stage 3   Load models       — Stella tokenizer + embedding model (once)
Stage 4   Index ALL docs    — chunk + embed every document (resumable, GPU)
Stage 5   Embed ALL queries — expand + embed every question's query (resumable, GPU)
Stage 6   Hybrid retrieve   — dense+BM25 retrieval for every question (resumable)
Stage 7   Rerank            — Jina rerank-v2 over Stage 6's candidates (resumable)
Stage 8   Select            — selection agent filters Stage 7's chunks (resumable)
Stage 9   Generate          — answer from Stage 8's selected chunks (resumable)
Stage 10  Score             — deterministic match + LLM judge fallback (resumable)
Stage 11  Summarize         — answer quality, retrieval Recall/MRR, tokens
```

### Resumability — important for a run this long

Stages 4-10 are each **resumable**: every document / query / question that already
has saved output on disk is skipped, and only new work is done. If the Colab
session disconnects partway through, just re-run the same cell — it picks up where
it left off. Failures are caught per-item and logged to `.log` files next to the
output, which are worth checking after a run finishes.

### Known scope decisions for this pass (see chat for the reasoning)
- **Gemini only.** DeepSeek V4 isn't wired in yet — `generation_model` is passed
  explicitly everywhere so adding it later is a config change, not a rewrite.
- **`gemini-3.1-flash-lite`, not `gemini-3.5-flash`.** The plain `gemini-3.5-flash`
  model hit its free-tier cap at just **20 requests/day** (`RESOURCE_EXHAUSTED`,
  `GenerateRequestsPerDayPerProjectPerModel-FreeTier`) — nowhere near enough for a
  150-question run. Flash-Lite variants get a much larger free-tier daily
  allowance. Check your actual live quota at ai.dev/rate-limit before assuming a
  full run fits in one day; Google no longer publishes static numbers.
- **Jina rerank-v2, not Voyage rerank-2.** Switched over to avoid Voyage's
  free-tier rate limits; Jina's actual limits haven't been measured yet, and its
  per-token pricing isn't confirmed (see `PRICING_PER_MILLION_TOKENS`) — fill that
  in before quoting a dollar total.
- **top_k_hybrid defaults to 10, not the paper's 20** — inherited from the old
  Voyage free-tier token cap; not yet re-verified against Jina. Bump back to 20
  once the reranker's real limits (and pricing) are known, and note the change
  either way.
- **Per-question latency is not measured by Stages 6-10** — see the trade-off
  note above.




---
## Stage 0 — Setup

Drive mount, repo clone/pull, API keys, output paths.

In [94]:
import os, sys, json, time, textwrap
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
if not (REPO_ROOT / "data" / "financebench_open_source.jsonl").exists():
    print(f"Repo not found at {REPO_ROOT} — cloning ...")
    !git clone https://github.com/shaliqsv/financebench-rag-thesis.git "{REPO_ROOT}"
else:
    # Plain `git pull` fails silently-ish ("You are not currently on a branch")
    # if this clone ever ends up in a detached-HEAD state (e.g. an interrupted
    # checkout) — checking out main explicitly every time makes this self-healing.
    # --no-rebase --no-edit: if a sync-to-GitHub cell below ever commits locally
    # but fails to push (e.g. GH_TOKEN missing), this clone and origin/main
    # diverge — plain `git pull` then refuses to guess how to reconcile them and
    # errors out. Auto-merging (not rebasing, so the local commit's hash is
    # preserved) with no editor prompt (which would hang in a notebook `!` cell)
    # makes that self-healing too.
    print(f"Repo already present at {REPO_ROOT} — syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

DATA_DIR  = REPO_ROOT / "data"
PDF_DIR   = REPO_ROOT / "pdfs"
INDEX_DIR = REPO_ROOT / "experiments" / "results" / "vector_rag_index"
QUERIES_DIR = INDEX_DIR / "queries"

# Stages 6-10 each save their own file instead of one combined results.jsonl —
# see the "What changed" note above for why.
RESULTS_DIR      = REPO_ROOT / "experiments" / "results"
HYBRID_PATH      = RESULTS_DIR / "vector_rag_stage_hybrid.jsonl"
RERANK_PATH      = RESULTS_DIR / "vector_rag_stage_rerank.jsonl"
SELECTION_PATH   = RESULTS_DIR / "vector_rag_stage_selection.jsonl"
GENERATION_PATH  = RESULTS_DIR / "vector_rag_stage_generation.jsonl"
SCORING_PATH     = RESULTS_DIR / "vector_rag_stage_scoring.jsonl"
COSTS_PATH       = RESULTS_DIR / "vector_rag_costs.jsonl"

sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT: {REPO_ROOT}")

load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY   = os.getenv("GOOGLE_API_KEY", "")
if GOOGLE_API_KEY:
    os.environ["GEMINI_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
GROQ_API_KEY     = os.getenv("GROQ_API_KEY", "")
JINA_API_KEY     = os.getenv("JINA_API_KEY", "")
GH_TOKEN         = os.getenv("GH_TOKEN", "")

print("\nAPI keys:")
for name, val in [("GOOGLE_API_KEY", GOOGLE_API_KEY), ("DEEPSEEK_API_KEY", DEEPSEEK_API_KEY),
                  ("GROQ_API_KEY", GROQ_API_KEY), ("JINA_API_KEY", JINA_API_KEY),
                  ("GH_TOKEN", GH_TOKEN)]:
    print(f"  {name:<20}: {'ok' if val else 'MISSING - fill in .env'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already present at /content/drive/MyDrive/financebench_project — syncing to latest main ...
M	experiments/results/vector_rag_costs.jsonl
D	experiments/results/vector_rag_stage_hybrid.jsonl
M	experiments/results/vector_rag_stage_rerank.jsonl
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Already up to date.
REPO_ROOT: /content/drive/MyDrive/financebench_project

API keys:
  GOOGLE_API_KEY      : ok
  DEEPSEEK_API_KEY    : MISSING - fill in .env
  GROQ_API_KEY        : ok
  JINA_API_KEY        : ok
  GH_TOKEN            : ok


### Git identity + push auth for Colab

The "sync to GitHub" cells below run inside the Colab container, which has no git
identity and no saved GitHub credentials. Without this cell, `git commit` fails
with "Author identity unknown" and `git push` fails with "could not read
Username" — and this failure is easy to miss, since the sync cells swallow the
commit error with `|| echo "(nothing new to commit)"`. That's exactly what
happened on an early Stage 4 (indexing) run: all 84 documents were indexed
correctly on Drive, but none of it reached GitHub.

Run this cell once per Colab session, right after Stage 0.
Requires `GH_TOKEN` in `.env` (a GitHub personal access token, repo scope —
see `.env.example`).

In [95]:
!git -C "{REPO_ROOT}" config user.email "shaliqv25@gmail.com"
!git -C "{REPO_ROOT}" config user.name "shaliqsv"

if GH_TOKEN:
    !git -C "{REPO_ROOT}" remote set-url origin https://{GH_TOKEN}@github.com/shaliqsv/financebench-rag-thesis.git
    print("git identity set, remote configured with token auth")
else:
    print("WARNING: GH_TOKEN missing from .env — sync-to-GitHub cells will fail to push. "
          "See .env.example for how to create one.")

git identity set, remote configured with token auth


### Sync helpers

One `_git_sync` helper, and one small wrapper per stage. Small, frequent syncs
(rather than one sync at the very end) mean a dropped Colab/VS Code connection on
a slow or unstable line loses at most one batch's worth of work, not the whole
run. Each stage's wrapper also syncs the cost log, since every stage appends to
it.

In [96]:
import subprocess

def _git_sync(paths: list[str], commit_msg: str, max_retries: int = 4, retry_delay: float = 5.0):
    """Stage add/commit/push, tolerant of the Drive-mounted repo's stale
    .git/index.lock (a concurrent git process, or Drive sync lag, briefly
    holding the lock). Retries a few times instead of raising — a sync
    hiccup should never crash a batch loop that has already made real
    progress; it just warns and leaves the sync for the next periodic call.

    If origin has moved ahead (e.g. pushed from another clone), the plain
    push is rejected as non-fast-forward; one pull+merge is attempted before
    retrying, since the results files are append-only jsonl and merge
    cleanly almost always."""
    add = None
    for attempt in range(max_retries):
        add = subprocess.run(["git", "-C", str(REPO_ROOT), "add", *paths], capture_output=True, text=True)
        if add.returncode == 0 or "index.lock" not in add.stderr:
            break
        print(f"    git sync: stale index.lock, retrying in {retry_delay:.0f}s ({attempt + 1}/{max_retries}) ...")
        time.sleep(retry_delay)

    if add.returncode != 0:
        print(f"    git sync FAILED (add): {add.stderr.strip()[:200]} — results are saved locally, will retry on next sync")
        return

    commit = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "commit", "-m", commit_msg],
        capture_output=True, text=True,
    )
    if commit.returncode != 0:
        print("    (nothing new to commit)")

    push = subprocess.run(["git", "-C", str(REPO_ROOT), "push", "origin", "main"], capture_output=True, text=True)
    if push.returncode != 0 and ("non-fast-forward" in push.stderr.lower() or "fetch first" in push.stderr.lower()):
        print("    git sync: origin has moved ahead, pulling and retrying push ...")
        pull = subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "origin", "main", "--no-rebase", "--no-edit"],
                               capture_output=True, text=True)
        if pull.returncode != 0:
            print(f"    git sync FAILED (pull after rejected push): {pull.stderr.strip()[:200]} — "
                  f"resolve manually in {REPO_ROOT}, results are saved locally")
            return
        push = subprocess.run(["git", "-C", str(REPO_ROOT), "push", "origin", "main"], capture_output=True, text=True)

    if push.returncode != 0:
        print(f"    git sync FAILED (push): {push.stderr.strip()[:200]} — results are saved locally, will retry on next sync")

def sync_index_to_github():
    _git_sync(["experiments/results/vector_rag_index"], "Sync indexed documents / query embeddings")

def sync_hybrid_to_github():
    _git_sync(["experiments/results/vector_rag_stage_hybrid.jsonl", "experiments/results/vector_rag_costs.jsonl"],
               "Sync hybrid retrieval stage + cost log")

def sync_rerank_to_github():
    _git_sync(["experiments/results/vector_rag_stage_rerank.jsonl", "experiments/results/vector_rag_costs.jsonl"],
               "Sync rerank stage + cost log")

def sync_selection_to_github():
    _git_sync(["experiments/results/vector_rag_stage_selection.jsonl", "experiments/results/vector_rag_costs.jsonl"],
               "Sync selection stage + cost log")

def sync_generation_to_github():
    _git_sync(["experiments/results/vector_rag_stage_generation.jsonl", "experiments/results/vector_rag_costs.jsonl"],
               "Sync generation stage + cost log")

def sync_scoring_to_github():
    _git_sync(["experiments/results/vector_rag_stage_scoring.jsonl", "experiments/results/vector_rag_costs.jsonl"],
               "Sync scoring stage + cost log")


In [97]:
%pip install -q pymupdf4llm tiktoken sentence-transformers rank_bm25 pyarrow requests "transformers==4.51.3" "sentence-transformers==3.3.1" google-genai --upgrade groq

---
## Stage 1 — Shared utilities

Retry/backoff for flaky API calls, token/cost logging, and the model clients —
previously `pipelines/vector_rag/api_utils.py` and `evaluation/cost_tracker.py`,
now defined here since every later stage uses them.

In [98]:
from functools import wraps
from dataclasses import asdict, dataclass

RATE_LIMIT_MARKERS = ("429", "rate limit", "resource_exhausted", "quota")


def _looks_like_rate_limit(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(marker in msg for marker in RATE_LIMIT_MARKERS)


def with_retry(max_retries: int = 6, base_delay: float = 5.0, max_delay: float = 120.0):
    """Exponential backoff, longer waits for rate-limit-shaped errors.

    Rate-limit errors back off starting at base_delay and double each retry
    (capped at max_delay); any other exception gets one short retry (network
    blip) before propagating, so real bugs still fail fast and loud instead
    of being silently retried into a 6x-slower version of the same crash.
    """
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            delay = base_delay
            last_exc = None
            for attempt in range(max_retries):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:  # noqa: BLE001 - deliberately broad, see docstring
                    last_exc = e
                    if _looks_like_rate_limit(e):
                        print(f"    rate limited ({e!s:.100s}), backing off {delay:.0f}s ...")
                        time.sleep(delay)
                        delay = min(delay * 2, max_delay)
                    elif attempt == 0:
                        print(f"    transient error ({e!s:.100s}), retrying once ...")
                        time.sleep(2.0)
                    else:
                        raise
            raise RuntimeError(f"{fn.__name__} failed after {max_retries} retries") from last_exc
        return wrapper
    return decorator


# Pricing is per 1M tokens. $0 for everything actually in use right now --
# these really are free (Gemini/Groq free tiers, Stella self-hosted on
# Colab's GPU), not a placeholder. deepseek-v4 and the Jina reranker stay
# None on purpose: deepseek-v4 isn't wired in yet and is NOT free; Jina's
# rerank pricing hasn't been confirmed yet (switched from Voyage rerank-2,
# see CLAUDE.md/Stage 7 note) -- both fail loudly (KeyError downstream) if
# used for real cost totals instead of silently logging $0.
PRICING_PER_MILLION_TOKENS: dict[str, dict[str, float | None]] = {
    "gemini-3.1-flash-lite": {"input": 0.0, "output": 0.0},   # free tier
    "deepseek-v4": {"input": None, "output": None},           # not wired in yet -- NOT free, fill in before use
    "jina-reranker-v2-base-multilingual": {"input": None, "output": None},  # pricing not confirmed -- fill in before use
    "stella-en-1.5b-v5": {"input": 0.0, "output": 0.0},       # self-hosted on Colab GPU, no API cost
    "openai/gpt-oss-120b": {"input": 0.0, "output": 0.0},     # judge, via Groq free tier
}


@dataclass
class UsageRecord:
    timestamp: float
    pipeline: str  # "vector_rag" | "vectorless_rag" | "long_context"
    stage: str  # e.g. "generation", "embedding", "rerank", "judge", "indexing"
    model: str
    doc_name: str | None
    financebench_id: str | None
    input_tokens: int
    output_tokens: int
    cost_usd: float | None


class CostTracker:
    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, pipeline, stage, model, input_tokens, output_tokens, doc_name=None, financebench_id=None):
        prices = PRICING_PER_MILLION_TOKENS.get(model)
        cost_usd = None
        if prices and prices["input"] is not None and prices["output"] is not None:
            cost_usd = (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000

        record = UsageRecord(
            timestamp=time.time(), pipeline=pipeline, stage=stage, model=model,
            doc_name=doc_name, financebench_id=financebench_id,
            input_tokens=input_tokens, output_tokens=output_tokens, cost_usd=cost_usd,
        )
        with self.log_path.open("a") as f:
            f.write(json.dumps(asdict(record)) + "\n")
        return record


cost_tracker = CostTracker(COSTS_PATH)
print(f"Logging costs to {cost_tracker.log_path}")

# Switched from gemini-3.5-flash: that model's free tier caps at just 20
# requests/day (RESOURCE_EXHAUSTED, GenerateRequestsPerDayPerProjectPerModel),
# nowhere near enough for a 150-question run. Flash-Lite variants get a much
# larger free-tier daily allowance. gemini-2.5-flash was deprecated for new
# API keys/projects before that, which is why this isn't just "2.5-flash".
GENERATION_MODEL = "gemini-3.1-flash-lite"
JUDGE_MODEL = "openai/gpt-oss-120b"     # via Groq free tier — see CLAUDE.md for why

from groq import Groq
from google import genai
import requests

genai_client = genai.Client()
judge_client   = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

print(f"Generation model: {GENERATION_MODEL}")
print(f"Judge model:      {JUDGE_MODEL}")


@with_retry()
def _generate(genai_client, model: str, prompt: str):
    """Shared low-level Gemini call, used by query expansion (Stage 5), the
    selection agent (Stage 8), and answer generation (Stage 9) — one retry-
    wrapped call site instead of three copies."""
    return genai_client.models.generate_content(model=model, contents=prompt)

Logging costs to /content/drive/MyDrive/financebench_project/experiments/results/vector_rag_costs.jsonl
Generation model: gemini-3.1-flash-lite
Judge model:      openai/gpt-oss-120b


---
## Stage 2 — Load FinanceBench data

All 150 questions across 84 unique source documents (some documents have multiple questions).

In [99]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

doc_names = sorted(df.doc_name.unique())

print(f"Total questions : {len(df)}")
print(f"Unique documents: {len(doc_names)}")
df[["financebench_id", "doc_name", "question"]].head(3)

Total questions : 150
Unique documents: 84


,financebench_id,doc_name,question
0,financebench_id_03029,3M_2018_10K,What is the FY2018 capital expenditure amount ...
1,financebench_id_04672,3M_2018_10K,Assume that you are a public equities analyst....
2,financebench_id_00499,3M_2022_10K,Is 3M a capital-intensive business based on FY...


---
## Stage 3 — Load Stella (tokenizer + embedding model)

Loaded once here and passed into every batch call below, instead of being reloaded
per document/query — reloading a 1.5B-parameter model 84+150 times would dominate
the runtime for no benefit.

In [100]:
from transformers import AutoTokenizer

STELLA_MODEL = "thomaskim1130/stella_en_400M_v5-FinanceRAG"
print(f"Loading tokenizer for {STELLA_MODEL} ...")
_stella_tokenizer = AutoTokenizer.from_pretrained(STELLA_MODEL, trust_remote_code=True)

def count_tokens(text: str) -> int:
    return len(_stella_tokenizer.encode(text, add_special_tokens=False))

print("Tokenizer ready")

Loading tokenizer for thomaskim1130/stella_en_400M_v5-FinanceRAG ...
Tokenizer ready


In [101]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("No GPU detected — indexing 84 documents on CPU will be very slow. "
          "In Colab: Runtime -> Change runtime type -> T4 GPU.")

t0 = time.time()
embed_model = SentenceTransformer(
    STELLA_MODEL,
    trust_remote_code=True,
    device=device,
    config_kwargs={"use_memory_efficient_attention": False, "unpad_inputs": False},
)
print(f"Loaded in {time.time() - t0:.1f}s — embedding dim {embed_model.get_sentence_embedding_dimension()}")

Device: cuda
Loaded in 4.6s — embedding dim 1024


---
## Stage 4 — Chunk + embed every document

**Chunking:** PDF -> per-page Markdown -> sentence-accumulated, page-bounded,
&le;512-token passages with a small sentence-level overlap at each chunk boundary.
Chunks never span a page boundary: FinanceBench's gold evidence is a page number
(`evidence_page_num`), so every chunk needs an unambiguous `page_num` to evaluate
retrieval against it.

**Indexing:** embed every chunk with Stella (dense) and build a BM25 index
(sparse, built fresh from the chunks at retrieval time in Stage 6 — cheap enough
that it isn't worth persisting). Saves `{doc_name}_chunks.parquet` and
`{doc_name}_dense.npy` to `INDEX_DIR`.

**Resumable** — a document already indexed is skipped, so re-running this cell
after an interruption only does the remaining work. On the first test document
(160 pages), parsing took ~150s and embedding ~145s, so budget roughly 5
minutes/document for a from-scratch run. **Auto-syncs to GitHub every 5
newly-indexed docs.**

In [102]:
import re
import pymupdf4llm

_SENT_SPLIT_RE = re.compile(r"(?<!\d)\.[ \n]+(?=[A-Z])")


def sent_tokenize(text: str) -> list[str]:
    sentences = _SENT_SPLIT_RE.split(text.strip())
    return [s.strip() for s in sentences if s.strip()]


def _greedy_token_split(units: list[str], count_tokens, max_tokens: int, joiner: str) -> list[str]:
    """Greedily group `units` into pieces <=max_tokens, rejoined with `joiner`.
    A single unit that alone exceeds max_tokens is recursed into word-level
    splitting; a single word that's still too big (essentially never happens
    in real filings) is accepted as-is rather than looping forever."""
    pieces, curr, curr_toks = [], [], 0
    for u in units:
        u_toks = count_tokens(u)
        if u_toks > max_tokens:
            if curr:
                pieces.append(joiner.join(curr))
                curr, curr_toks = [], 0
            sub_units = u.split()
            if len(sub_units) > 1:
                pieces.extend(_greedy_token_split(sub_units, count_tokens, max_tokens, " "))
            else:
                pieces.append(u)
            continue
        if curr_toks + u_toks > max_tokens and curr:
            pieces.append(joiner.join(curr))
            curr, curr_toks = [], 0
        curr.append(u)
        curr_toks += u_toks
    if curr:
        pieces.append(joiner.join(curr))
    return pieces


def split_oversized_sentence(sent: str, count_tokens, max_tokens: int) -> list[str]:
    """A 'sentence' from sent_tokenize that alone exceeds max_tokens is almost
    always a table block: rows of numbers with no period-then-capital-letter
    boundary for the regex above to find (confirmed: 11.5% of chunks across
    the 84 indexed docs came out oversized this way, up to ~6x the target,
    concentrated in filings like JPMorgan and MGM Resorts with dense tables).
    Falls back to splitting on newlines first (markdown table rows are
    newline-separated), then words if a single line is still too long, so no
    chunk chunk_page produces can ever exceed max_tokens."""
    lines = [l for l in sent.split("\n") if l.strip()]
    if len(lines) > 1:
        return _greedy_token_split(lines, count_tokens, max_tokens, "\n")
    return _greedy_token_split(sent.split(), count_tokens, max_tokens, " ")


def chunk_page(page_text, page_num, doc_name, count_tokens, max_tokens=512, overlap_tokens=50):
    """Split one page's markdown into <=max_tokens passages, page-bounded.

    Algorithm: greedily accumulate sentences until the token budget would be
    exceeded, flush, then seed the next chunk with the last ~overlap_tokens
    of the flushed chunk so boundary sentences appear in both chunks. Any
    single sentence that alone exceeds max_tokens (a table block, see
    split_oversized_sentence) is expanded into multiple sub-units before
    accumulation, so every unit fed into the loop below already fits the
    budget on its own -- no chunk this function returns can exceed max_tokens.
    """
    if not page_text.strip():
        return []

    sentences = []
    for s in sent_tokenize(page_text):
        s_toks = count_tokens(s)
        if s_toks > max_tokens:
            pieces = split_oversized_sentence(s, count_tokens, max_tokens)
            print(f"    [chunk_page] {doc_name} p{page_num}: oversized block "
                  f"({s_toks} tok, target {max_tokens}) -> split into {len(pieces)} pieces "
                  f"({[count_tokens(p) for p in pieces]} tok each)")
            sentences.extend(pieces)
        else:
            sentences.append(s)

    chunks = []
    curr_sents, curr_tokens = [], 0

    for sent in sentences:
        sent_toks = count_tokens(sent)

        if curr_tokens + sent_toks > max_tokens and curr_sents:
            chunks.append({
                "doc_name": doc_name, "page_num": page_num, "chunk_idx": len(chunks),
                "text": " ".join(curr_sents), "token_count": curr_tokens,
            })
            overlap_sents, overlap_toks = [], 0
            for s in reversed(curr_sents):
                s_toks = count_tokens(s)
                if overlap_toks + s_toks <= overlap_tokens:
                    overlap_sents.insert(0, s)
                    overlap_toks += s_toks
                else:
                    break
            curr_sents = overlap_sents + [sent]
            curr_tokens = overlap_toks + sent_toks
        else:
            curr_sents.append(sent)
            curr_tokens += sent_toks

    if curr_sents:
        chunks.append({
            "doc_name": doc_name, "page_num": page_num, "chunk_idx": len(chunks),
            "text": " ".join(curr_sents), "token_count": curr_tokens,
        })
    return chunks


def parse_and_chunk_document(pdf_path, doc_name, count_tokens, max_tokens=512, overlap_tokens=50, verbose=True):
    """Full Stage 4 parse+chunk for one document: PDF -> per-page markdown -> chunks_df.

    pymupdf4llm's page_number is 1-indexed; stored page_num is 0-indexed so it
    matches FinanceBench's evidence_page_num convention exactly.
    """
    t0 = time.time()
    md_pages = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
    if verbose:
        print(f"  parsed {len(md_pages)} pages in {time.time() - t0:.1f}s")

    all_chunks = []
    for page_dict in md_pages:
        pg_num = page_dict["metadata"]["page_number"] - 1  # -> 0-indexed
        all_chunks.extend(chunk_page(page_dict["text"], pg_num, doc_name, count_tokens, max_tokens, overlap_tokens))

    chunks_df = pd.DataFrame(all_chunks).reset_index(drop=True)
    if chunks_df.empty:
        raise ValueError(f"{doc_name}: no chunks produced (empty/unparseable PDF?)")

    chunks_df["text"] = chunks_df["text"].str.strip()
    chunks_df["chunk_id"] = chunks_df.apply(lambda r: f"{r.doc_name}__p{r.page_num:04d}_c{r.chunk_idx:02d}", axis=1)
    return chunks_df

In [103]:
import traceback
import numpy as np
from rank_bm25 import BM25Okapi

_BM25_TOKEN_RE = re.compile(r"[a-z0-9$][a-z0-9.,%$-]*")


def bm25_tokenize(text: str) -> list[str]:
    return _BM25_TOKEN_RE.findall(text.lower())


def build_bm25(chunks_df) -> BM25Okapi:
    return BM25Okapi([bm25_tokenize(t) for t in chunks_df["text"]])


def format_query(query: str) -> str:
    return f"Instruct: Given a web search query, retrieve relevant passages that answer the query.\nQuery: {query}"


def index_paths(index_dir, doc_name):
    return index_dir / f"{doc_name}_chunks.parquet", index_dir / f"{doc_name}_dense.npy"


def is_indexed(index_dir, doc_name) -> bool:
    chunks_path, dense_path = index_paths(index_dir, doc_name)
    return chunks_path.exists() and dense_path.exists()


def load_index(index_dir, doc_name):
    chunks_path, dense_path = index_paths(index_dir, doc_name)
    chunks_df = pd.read_parquet(chunks_path)
    dense_embeddings = np.load(dense_path)
    assert len(chunks_df) == dense_embeddings.shape[0], f"{doc_name}: chunks/embeddings length mismatch"
    return chunks_df, dense_embeddings


def index_document(pdf_path, doc_name, embed_model, count_tokens, index_dir, batch_size=8):
    """Parse + chunk + embed one document, save to index_dir, return the result.

    Caller is responsible for skip-if-already-indexed (see index_all_documents)
    so this function always does the full (slow) work when called directly.
    """
    chunks_df = parse_and_chunk_document(pdf_path, doc_name, count_tokens)

    dense_embeddings = embed_model.encode(
        chunks_df["text"].tolist(), batch_size=batch_size, show_progress_bar=False,
        normalize_embeddings=True, convert_to_numpy=True,
    )

    index_dir.mkdir(parents=True, exist_ok=True)
    chunks_path, dense_path = index_paths(index_dir, doc_name)
    np.save(dense_path, dense_embeddings)
    chunks_df.to_parquet(chunks_path)
    return chunks_df, dense_embeddings


def index_all_documents(doc_names, pdf_dir, embed_model, count_tokens, index_dir, batch_size=8,
                         sync_every=None, sync_fn=None):
    """Resumable batch driver: index every doc in doc_names not already saved.

    A single bad PDF must not abort a 7+ hour unattended run, so failures are
    caught, logged to index_dir/indexing_errors.log, and skipped — check that
    file after the run finishes to see whether the failure count is nonzero.
    Safe to re-run after an interruption: already-indexed docs are skipped.
    """
    errors_log = index_dir / "indexing_errors.log"
    results = {"indexed": [], "skipped": [], "failed": []}
    synced_through = 0

    for i, doc_name in enumerate(doc_names, start=1):
        if is_indexed(index_dir, doc_name):
            results["skipped"].append(doc_name)
            print(f"[{i}/{len(doc_names)}] {doc_name}: already indexed, skipping")
            continue

        pdf_path = pdf_dir / f"{doc_name}.pdf"
        print(f"[{i}/{len(doc_names)}] {doc_name}: indexing ...")
        t0 = time.time()
        try:
            chunks_df, _ = index_document(pdf_path, doc_name, embed_model, count_tokens, index_dir, batch_size)
            print(f"    done in {time.time() - t0:.1f}s — {len(chunks_df)} chunks")
            results["indexed"].append(doc_name)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{doc_name}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(doc_name)

        if sync_fn is not None and sync_every and len(results["indexed"]) - synced_through >= sync_every:
            print(f"    -- syncing after {len(results['indexed'])} newly indexed docs --")
            sync_fn()
            synced_through = len(results["indexed"])

    if sync_fn is not None and len(results["indexed"]) > synced_through:
        print(f"    -- final sync ({len(results['indexed'])} newly indexed docs total) --")
        sync_fn()

    print(f"\nIndexing summary: {len(results['indexed'])} indexed, "
          f"{len(results['skipped'])} already done, {len(results['failed'])} failed")
    if results["failed"]:
        print(f"Failed docs (see {errors_log}): {results['failed']}")
    return results

In [104]:
index_results = index_all_documents(
    doc_names=doc_names,
    pdf_dir=PDF_DIR,
    embed_model=embed_model,
    count_tokens=count_tokens,
    index_dir=INDEX_DIR,
    sync_every=5,
    sync_fn=sync_index_to_github,
)

[1/84] 3M_2018_10K: already indexed, skipping
[2/84] 3M_2022_10K: already indexed, skipping
[3/84] 3M_2023Q2_10Q: already indexed, skipping
[4/84] ACTIVISIONBLIZZARD_2019_10K: already indexed, skipping
[5/84] ADOBE_2015_10K: already indexed, skipping
[6/84] ADOBE_2016_10K: already indexed, skipping
[7/84] ADOBE_2017_10K: already indexed, skipping
[8/84] ADOBE_2022_10K: already indexed, skipping
[9/84] AES_2022_10K: already indexed, skipping
[10/84] AMAZON_2017_10K: already indexed, skipping
[11/84] AMAZON_2019_10K: already indexed, skipping
[12/84] AMCOR_2020_10K: already indexed, skipping
[13/84] AMCOR_2022_8K_dated-2022-07-01: already indexed, skipping
[14/84] AMCOR_2023Q2_10Q: already indexed, skipping
[15/84] AMCOR_2023Q4_EARNINGS: already indexed, skipping
[16/84] AMCOR_2023_10K: already indexed, skipping
[17/84] AMD_2015_10K: already indexed, skipping
[18/84] AMD_2022_10K: already indexed, skipping
[19/84] AMERICANEXPRESS_2022_10K: already indexed, skipping
[20/84] AMERICANWATERW

In [105]:
index_results

{'indexed': [],
 'skipped': ['3M_2018_10K',
  '3M_2022_10K',
  '3M_2023Q2_10Q',
  'ACTIVISIONBLIZZARD_2019_10K',
  'ADOBE_2015_10K',
  'ADOBE_2016_10K',
  'ADOBE_2017_10K',
  'ADOBE_2022_10K',
  'AES_2022_10K',
  'AMAZON_2017_10K',
  'AMAZON_2019_10K',
  'AMCOR_2020_10K',
  'AMCOR_2022_8K_dated-2022-07-01',
  'AMCOR_2023Q2_10Q',
  'AMCOR_2023Q4_EARNINGS',
  'AMCOR_2023_10K',
  'AMD_2015_10K',
  'AMD_2022_10K',
  'AMERICANEXPRESS_2022_10K',
  'AMERICANWATERWORKS_2020_10K',
  'AMERICANWATERWORKS_2021_10K',
  'AMERICANWATERWORKS_2022_10K',
  'BESTBUY_2017_10K',
  'BESTBUY_2019_10K',
  'BESTBUY_2023_10K',
  'BESTBUY_2024Q2_10Q',
  'BLOCK_2016_10K',
  'BLOCK_2020_10K',
  'BOEING_2018_10K',
  'BOEING_2022_10K',
  'COCACOLA_2017_10K',
  'COCACOLA_2021_10K',
  'COCACOLA_2022_10K',
  'CORNING_2020_10K',
  'CORNING_2021_10K',
  'CORNING_2022_10K',
  'COSTCO_2021_10K',
  'CVSHEALTH_2018_10K',
  'CVSHEALTH_2022_10K',
  'FOOTLOCKER_2022_8K_dated-2022-05-20',
  'FOOTLOCKER_2022_8K_dated_2022-08-19',

### Sync indexed documents back to GitHub

`INDEX_DIR` lives on Drive, which only this Colab session can see. Committing it
to the repo means a plain `git pull` on your **local machine** fetches every
document already indexed here. The cell above auto-syncs every 5 docs; kept here
as a one-off you can run any time.

In [106]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_index
!git -C "{REPO_ROOT}" commit -m "Sync indexed documents" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   experiments/results/vector_rag_costs.jsonl
	deleted:    experiments/results/vector_rag_stage_hybrid.jsonl
	modified:   experiments/results/vector_rag_stage_rerank.jsonl

no changes added to commit (use "git add" and/or "git commit -a")
(nothing new to commit)
Everything up-to-date


---
## Stage 5 — Expand + embed every question's query

For each of the 150 questions: ask Gemini to expand the question into a
retrieval-friendly query, then embed that expanded query with Stella. Saves
`{financebench_id}__expanded.npy` / `.json` to `QUERIES_DIR`. **Resumable**, and
**auto-syncs to GitHub every 20 newly-embedded queries**.

Still needs the GPU-backed Stella model, which is why it runs here rather than
in a later stage.

In [107]:
EXPANSION_PROMPT = """You are helping a retrieval system find the right passage in a financial \
filing (10-K/10-Q). Rewrite the question below into a short expanded search query: include likely \
synonyms, exact financial line-item names, and related terms that would appear near the answer in \
the filing. Return ONLY the expanded query text - no preamble, no explanation, no quotes.

Question: {question}"""


def expand_query(genai_client, question: str, model: str):
    resp = _generate(genai_client, model, EXPANSION_PROMPT.format(question=question))
    return resp.text.strip(), resp.usage_metadata

In [108]:
# Gemini free tier caps requests/minute; expand_query makes one call per
# question, so pacing each iteration to at least this long keeps the loop under
# that cap instead of bursting and eating with_retry's 429 backoff penalty on
# every other question.
MIN_SECONDS_BETWEEN_CALLS = 6.5


def query_paths(queries_dir, financebench_id):
    return (queries_dir / f"{financebench_id}__expanded.npy", queries_dir / f"{financebench_id}__expanded.json")


def is_query_embedded(queries_dir, financebench_id) -> bool:
    vec_path, meta_path = query_paths(queries_dir, financebench_id)
    return vec_path.exists() and meta_path.exists()


def embed_all_queries(df, genai_client, embed_model, expansion_model, queries_dir, cost_tracker,
                       sync_every=None, sync_fn=None):
    """Resumable batch driver: expand + embed every question's query.

    One failed question (e.g. a malformed expansion response) is logged and
    skipped rather than aborting the whole batch — check
    queries_dir/embedding_errors.log afterward for a nonzero failure count.
    """
    queries_dir.mkdir(parents=True, exist_ok=True)
    errors_log = queries_dir / "embedding_errors.log"
    results = {"embedded": [], "skipped": [], "failed": []}
    synced_through = 0

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if is_query_embedded(queries_dir, fb_id):
            results["skipped"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: expanding + embedding ...")
        t0 = time.time()
        try:
            expanded_query, usage = expand_query(genai_client, row.question, expansion_model)
            cost_tracker.log(pipeline="vector_rag", stage="query_expansion", model=expansion_model,
                              input_tokens=usage.prompt_token_count, output_tokens=usage.candidates_token_count,
                              doc_name=row.doc_name, financebench_id=fb_id)
            print(f"    question: {row.question[:90]!r}")
            print(f"    expanded: {expanded_query[:90]!r}")

            query_vec = embed_model.encode(format_query(expanded_query), normalize_embeddings=True)

            vec_path, meta_path = query_paths(queries_dir, fb_id)
            np.save(vec_path, query_vec)
            meta_path.write_text(json.dumps({
                "financebench_id": fb_id, "doc_name": row.doc_name, "tag": "expanded", "query_text": expanded_query,
            }, indent=2))

            print(f"    done in {time.time() - t0:.1f}s")
            results["embedded"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

        time.sleep(max(0.0, MIN_SECONDS_BETWEEN_CALLS - (time.time() - t0)))

        if sync_fn is not None and sync_every and len(results["embedded"]) - synced_through >= sync_every:
            print(f"    -- syncing after {len(results['embedded'])} newly embedded queries --")
            sync_fn()
            synced_through = len(results["embedded"])

    if sync_fn is not None and len(results["embedded"]) > synced_through:
        print(f"    -- final sync ({len(results['embedded'])} newly embedded queries total) --")
        sync_fn()

    print(f"\nQuery embedding summary: {len(results['embedded'])} embedded, "
          f"{len(results['skipped'])} already done, {len(results['failed'])} failed")
    if results["failed"]:
        print(f"Failed questions (see {errors_log}): {results['failed']}")
    return results

In [109]:
print(genai.__version__)

2.22.0


In [110]:
query_embed_results = embed_all_queries(
    df=df,
    genai_client=genai_client,
    embed_model=embed_model,
    expansion_model=GENERATION_MODEL,
    queries_dir=QUERIES_DIR,
    cost_tracker=cost_tracker,
    sync_every=20,
    sync_fn=sync_index_to_github,
)


Query embedding summary: 0 embedded, 150 already done, 0 failed


### Sync query embeddings back to GitHub

`QUERIES_DIR` is a subfolder of `INDEX_DIR`, so this covers the newly embedded
queries too. The cell above auto-syncs every 20 queries; kept here as a manual
one-off trigger.

In [111]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_index
!git -C "{REPO_ROOT}" commit -m "Sync query embeddings" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   experiments/results/vector_rag_costs.jsonl
	deleted:    experiments/results/vector_rag_stage_hybrid.jsonl
	modified:   experiments/results/vector_rag_stage_rerank.jsonl

no changes added to commit (use "git add" and/or "git commit -a")
(nothing new to commit)
Everything up-to-date


---
## Stage 6 — Hybrid retrieval (dense + BM25) for every question

`hybrid_score = alpha * dense_cosine + (1 - alpha) * bm25_minmax_normalized`,
alpha=0.85 per Kim et al. (dense weighted higher). BM25's raw scores are
unbounded term-frequency sums while dense cosine is already in [0, 1] for
normalized embeddings, so BM25 is min-max normalized over the candidate set
first — otherwise dense would dominate regardless of alpha.

Retrieval quality (Recall@k / MRR@k against FinanceBench's gold
`evidence_page_num`) is computed at the **page** level, not the chunk level,
since that's what the gold annotations are.

This is the first of the five split stages: pure numpy/BM25 math, no paid API,
no GPU — the cheapest and fastest of the five, and a good place to sanity-check
retrieval quality before spending any generation-model budget downstream.
**Resumable**, saves one line per question to `HYBRID_PATH`.

In [112]:
K_VALUES = (5, 10, 15)


def recall_at_k(ranked_pages, gold_pages, k) -> float:
    """Fraction of gold pages present anywhere in the top-k ranked pages."""
    gold_set = set(gold_pages)
    if not gold_set:
        raise ValueError("gold_pages must be non-empty")
    retrieved = set(ranked_pages[:k])
    return len(retrieved & gold_set) / len(gold_set)


def reciprocal_rank_at_k(ranked_pages, gold_pages, k) -> float:
    """1/rank of the first gold-page hit within the top-k ranked pages, else 0."""
    gold_set = set(gold_pages)
    for rank, page in enumerate(ranked_pages[:k], start=1):
        if page in gold_set:
            return 1.0 / rank
    return 0.0


@dataclass
class RetrievalMetrics:
    financebench_id: str
    doc_name: str
    stage: str  # e.g. "hybrid_top10" or "rerank_top10" — which pipeline stage this measures
    recall_at_k: dict
    mrr_at_k: dict


def compute_retrieval_metrics(ranked_pages, gold_pages, financebench_id, doc_name, stage, k_values=K_VALUES):
    gold_set = set(gold_pages)
    return RetrievalMetrics(
        financebench_id=financebench_id, doc_name=doc_name, stage=stage,
        recall_at_k={k: recall_at_k(ranked_pages, gold_set, k) for k in k_values},
        mrr_at_k={k: reciprocal_rank_at_k(ranked_pages, gold_set, k) for k in k_values},
    )

In [113]:
ALPHA_DENSE = 0.85  # dense weight, per Kim et al.


def hybrid_retrieve(chunks_df, dense_embeddings, bm25_index, query_vec, expanded_query, top_k=20, alpha=ALPHA_DENSE):
    dense_scores = dense_embeddings @ query_vec
    bm25_scores_raw = bm25_index.get_scores(bm25_tokenize(expanded_query))

    bm25_min, bm25_max = bm25_scores_raw.min(), bm25_scores_raw.max()
    bm25_scores_norm = (bm25_scores_raw - bm25_min) / (bm25_max - bm25_min + 1e-9)

    hybrid_scores = alpha * dense_scores + (1 - alpha) * bm25_scores_norm

    top_idx = np.argsort(-hybrid_scores)[:top_k]
    top = chunks_df.iloc[top_idx].copy()
    top["hybrid_score"] = hybrid_scores[top_idx]
    top["dense_score"] = dense_scores[top_idx]
    top["bm25_score"] = bm25_scores_raw[top_idx]
    return top

In [114]:
def _load_jsonl(path) -> list[dict]:
    if not path.exists():
        return []
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def _load_completed_ids(path) -> set:
    return {r["financebench_id"] for r in _load_jsonl(path)}


def _load_stage_records(path) -> dict:
    """financebench_id -> record, for reading a previous stage's output."""
    return {r["financebench_id"]: r for r in _load_jsonl(path)}

In [118]:
def run_hybrid_retrieval_all(df, index_dir, queries_dir, out_path, top_k=20, alpha=ALPHA_DENSE,
                              sync_every=None, sync_fn=None):
    """Stage 6: hybrid dense+BM25 retrieval for every question, saved separately
    from every later stage so retrieval quality can be inspected on its own.
    Resumable: skips any financebench_id already in out_path."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "hybrid_errors.log"
    completed = _load_completed_ids(out_path)
    synced_through = 0
    doc_cache = {}
    results = {"done": [], "skipped": [], "failed": [], "missing_index": [], "missing_query": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if not is_indexed(index_dir, row.doc_name):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — {row.doc_name} not indexed yet")
            results["missing_index"].append(fb_id)
            continue
        if not is_query_embedded(queries_dir, fb_id):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — query not embedded yet")
            results["missing_query"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: hybrid retrieving ...")
        try:
            if row.doc_name not in doc_cache:
                chunks_df, dense_embeddings = load_index(index_dir, row.doc_name)
                bm25_index = build_bm25(chunks_df)
                doc_cache[row.doc_name] = (chunks_df, dense_embeddings, bm25_index)
            chunks_df, dense_embeddings, bm25_index = doc_cache[row.doc_name]

            vec_path, meta_path = query_paths(queries_dir, fb_id)
            query_vec = np.load(vec_path)
            expanded_query = json.loads(meta_path.read_text())["query_text"]
            gold_pages = [e["evidence_page_num"] for e in row.evidence]

            top_hybrid = hybrid_retrieve(chunks_df, dense_embeddings, bm25_index, query_vec, expanded_query, top_k=top_k, alpha=alpha)
            metrics = compute_retrieval_metrics(
                ranked_pages=top_hybrid["page_num"].tolist(), gold_pages=gold_pages,
                financebench_id=fb_id, doc_name=row.doc_name, stage=f"hybrid_top{top_k}",
            )
            top3 = list(zip(top_hybrid["page_num"].tolist()[:3], top_hybrid["hybrid_score"].round(3).tolist()[:3]))
            print(f"    gold pages: {gold_pages}  top-3 retrieved (page, score): {top3}")
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "chunk_ids": top_hybrid["chunk_id"].tolist(),
                "page_nums": top_hybrid["page_num"].tolist(),
                "hybrid_scores": top_hybrid["hybrid_score"].tolist(),
                "recall_at_k": metrics.recall_at_k, "mrr_at_k": metrics.mrr_at_k,
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            print(f"    done — recall@20={metrics.recall_at_k[10]:.2f}")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

        if sync_fn is not None and sync_every and len(results["done"]) - synced_through >= sync_every:
            print(f"    -- syncing after {len(results['done'])} newly retrieved questions --")
            sync_fn()
            synced_through = len(results["done"])

    if sync_fn is not None and len(results["done"]) > synced_through:
        sync_fn()

    print(f"\nHybrid retrieval summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_index'])} missing index, "
          f"{len(results['missing_query'])} missing query embedding")
    if results["failed"]:
        print(f"Failed questions (see {errors_log}): {results['failed']}")
    return results

In [119]:
HYBRID_PATH


PosixPath('/content/drive/MyDrive/financebench_project/experiments/results/vector_rag_stage_hybrid.jsonl')

In [125]:
hybrid_results = run_hybrid_retrieval_all(
    df=df,
    index_dir=INDEX_DIR,
    queries_dir=QUERIES_DIR,
    out_path=HYBRID_PATH,
    top_k=20,
    sync_every=15,
    sync_fn=sync_hybrid_to_github,
)

[1/150] financebench_id_03029: hybrid retrieving ...
    gold pages: [59]  top-3 retrieved (page, score): [(46, 0.726), (38, 0.696), (45, 0.694)]
    done — recall@20=1.00
[2/150] financebench_id_04672: hybrid retrieving ...
    gold pages: [57]  top-3 retrieved (page, score): [(57, 0.713), (40, 0.691), (38, 0.626)]
    done — recall@20=1.00
[3/150] financebench_id_00499: hybrid retrieving ...
    gold pages: [47, 49, 51]  top-3 retrieved (page, score): [(38, 0.72), (38, 0.719), (33, 0.669)]
    done — recall@20=0.00
[4/150] financebench_id_01226: hybrid retrieving ...
    gold pages: [26]  top-3 retrieved (page, score): [(26, 0.762), (18, 0.735), (20, 0.715)]
    done — recall@20=1.00
[5/150] financebench_id_01865: hybrid retrieving ...
    gold pages: [24]  top-3 retrieved (page, score): [(32, 0.671), (18, 0.663), (20, 0.663)]
    done — recall@20=0.00
[6/150] financebench_id_00807: hybrid retrieving ...
    gold pages: [4]  top-3 retrieved (page, score): [(4, 0.68), (69, 0.668), (70

### Sync hybrid retrieval results back to GitHub

Auto-syncs every 15 questions above; kept here as a manual one-off.

In [121]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_stage_hybrid.jsonl experiments/results/vector_rag_costs.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync hybrid retrieval stage" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
(nothing new to commit)
Everything up-to-date


---
## Stage 7 — Rerank Stage 6's candidates with Jina rerank-v2

Re-scores each question's top-10 hybrid candidates and keeps the top 10 in
reranked order. **top_k defaults to 10, not the paper's 20** — this was
originally set to stay under Voyage rerank-2's free-tier ~10K-tokens/minute
cap before the switch to Jina (`jina-reranker-v2-base-multilingual`, see
CLAUDE.md); Jina's actual limits haven't been measured yet, so this default
is carried over rather than re-verified. Revisit once real call volume shows
Jina's actual limits, and note the change (and which reranker was used) in
the thesis methodology section either way.

Reads chunk text back out of the document index by `chunk_id` (Stage 6 only
stored ids/scores, not full text, to keep that file small). **Resumable** —
needs Stage 6 to have produced a hybrid record for the question first.

In [126]:
# Switched from Voyage rerank-2 to Jina's rerank API (jina-reranker-v2-base-
# multilingual, via a plain REST call -- no SDK needed) -- see CLAUDE.md/Stage
# 7 note. Voyage's free-tier caps (3 RPM / 10K TPM) were measured empirically
# from its RateLimitError bodies; Jina's actual limits haven't been observed
# yet, so there's no equivalent fixed pre-emptive sleep here. with_retry's
# exponential backoff (above) handles 429s adaptively instead. Tighten this
# once real 429 behavior against Jina is seen.
JINA_RERANK_URL = "https://api.jina.ai/v1/rerank"

# Carried over from the old Voyage budget-trim logic (see git history) as a
# defensive default, NOT verified against Jina's actual per-request limits.
JINA_TOKEN_BUDGET = 9000


@with_retry(max_retries=5, base_delay=10.0, max_delay=120.0)
def _jina_rerank(jina_api_key, query, documents, model, top_k):
    resp = requests.post(
        JINA_RERANK_URL,
        headers={"Authorization": f"Bearer {jina_api_key}", "Content-Type": "application/json"},
        json={"model": model, "query": query, "documents": documents, "top_n": top_k},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()


def rerank(jina_api_key, query, candidates_df, top_k=10, model="jina-reranker-v2-base-multilingual"):
    """Returns (reranked_df, total_tokens) — total_tokens is for cost logging.

    Trims the lowest-ranked candidates first if the full set would exceed
    JINA_TOKEN_BUDGET — candidates_df arrives in hybrid-score order, so
    trimming from the tail keeps the most relevant chunks. Without this, a
    document with a few oversized table chunks (see financebench_id_04171,
    an MGM Resorts 10-K with 10 hybrid candidates totaling 11,822 tokens on
    their own) sends one call that risks being rejected outright."""
    trimmed = candidates_df
    if "token_count" in trimmed.columns:
        within_budget = trimmed["token_count"].cumsum() <= JINA_TOKEN_BUDGET
        if not within_budget.all():
            trimmed = trimmed.iloc[:max(1, within_budget.sum())]

    result = _jina_rerank(jina_api_key, query, trimmed["text"].tolist(), model, top_k)

    order = [r["index"] for r in result["results"]]
    scores = [r["relevance_score"] for r in result["results"]]
    total_tokens = result.get("usage", {}).get("total_tokens", 0)

    top = trimmed.iloc[order].copy()
    top["rerank_score"] = scores
    return top, total_tokens

In [127]:
def run_rerank_all(df, index_dir, queries_dir, hybrid_path, out_path, top_k=10, sync_every=None, sync_fn=None):
    """Stage 7: Jina rerank-v2 over each question's Stage 6 hybrid candidates."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "rerank_errors.log"
    hybrid_records = _load_stage_records(hybrid_path)
    completed = _load_completed_ids(out_path)
    synced_through = 0
    doc_cache = {}
    results = {"done": [], "skipped": [], "failed": [], "missing_hybrid": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if fb_id not in hybrid_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — no hybrid retrieval yet")
            results["missing_hybrid"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: reranking ...")
        try:
            hyb = hybrid_records[fb_id]
            if row.doc_name not in doc_cache:
                chunks_df, _ = load_index(index_dir, row.doc_name)
                doc_cache[row.doc_name] = chunks_df.set_index("chunk_id")
            chunks_by_id = doc_cache[row.doc_name]
            candidates_df = chunks_by_id.loc[hyb["chunk_ids"]].reset_index()

            _, meta_path = query_paths(queries_dir, fb_id)
            expanded_query = json.loads(meta_path.read_text())["query_text"]
            gold_pages = [e["evidence_page_num"] for e in row.evidence]

            top_rerank, rerank_tokens = rerank(JINA_API_KEY, expanded_query, candidates_df, top_k=top_k)
            cost_tracker.log(pipeline="vector_rag", stage="rerank", model="jina-reranker-v2-base-multilingual",
                              input_tokens=rerank_tokens, output_tokens=0, doc_name=row.doc_name, financebench_id=fb_id)

            metrics = compute_retrieval_metrics(
                ranked_pages=top_rerank["page_num"].tolist(), gold_pages=gold_pages,
                financebench_id=fb_id, doc_name=row.doc_name, stage=f"rerank_top{top_k}",
            )
            top3 = list(zip(top_rerank["page_num"].tolist()[:3], [round(s, 3) for s in top_rerank["rerank_score"].tolist()[:3]]))
            print(f"    gold pages: {gold_pages}  top-3 reranked (page, score): {top3}")
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "chunk_ids": top_rerank["chunk_id"].tolist(),
                "page_nums": top_rerank["page_num"].tolist(),
                "rerank_scores": top_rerank["rerank_score"].tolist(),
                "recall_at_k": metrics.recall_at_k, "mrr_at_k": metrics.mrr_at_k,
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            print(f"    done — recall@10={metrics.recall_at_k[10]:.2f}")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

        if sync_fn is not None and sync_every and len(results["done"]) - synced_through >= sync_every:
            print(f"    -- syncing after {len(results['done'])} newly reranked questions --")
            sync_fn()
            synced_through = len(results["done"])

    if sync_fn is not None and len(results["done"]) > synced_through:
        sync_fn()

    print(f"\nRerank summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_hybrid'])} missing hybrid retrieval")
    if results["failed"]:
        print(f"Failed questions (see {errors_log}): {results['failed']}")
    return results

In [128]:
rerank_results = run_rerank_all(
    df=df,
    index_dir=INDEX_DIR,
    queries_dir=QUERIES_DIR,
    hybrid_path=HYBRID_PATH,
    out_path=RERANK_PATH,
    top_k=10,
    sync_every=15,
    sync_fn=sync_rerank_to_github,
)


Rerank summary: 0 done, 150 already done, 0 failed, 0 missing hybrid retrieval


### Sync rerank results back to GitHub

Auto-syncs every 15 questions above; kept here as a manual one-off.

In [129]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_stage_rerank.jsonl experiments/results/vector_rag_costs.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync rerank stage" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
(nothing new to commit)
Everything up-to-date


---
## Stage 8 — Selection agent

The GAR paper's selection step: ask the generation model to filter Stage 7's
reranked top-10 chunks down to just what's needed to answer the question, before
those chunks are spent on generation. If the model returns an empty selection
(it's asked to keep at least one, but that instruction isn't trusted blindly),
all candidate chunks are kept instead — an empty selection would otherwise
silently generate from zero context, producing a spurious "Failure to Answer"
that looks like a retrieval miss when it's actually a selection-stage bug.

**Resumable** — needs Stage 7 to have produced a rerank record for the question
first.

In [130]:
SELECTION_PROMPT = """You are filtering retrieved passages from a financial filing before they are \
passed to an answer-generation model. Given the question and candidate passages below, return ONLY \
the passage IDs that are actually needed to answer the question, as a JSON array of strings \
(e.g. ["id1", "id2"]). Drop passages that are irrelevant or redundant. Keep at least one passage. \
Return nothing except the JSON array.

Question: {question}

Candidate passages:
{passages}"""


def format_passages(df) -> str:
    return "\n\n".join(f"[{r.chunk_id}] (page {r.page_num})\n{r.text}" for _, r in df.iterrows())


def select_chunks(genai_client, question, candidates_df, model):
    resp = _generate(genai_client, model, SELECTION_PROMPT.format(question=question, passages=format_passages(candidates_df)))
    raw = resp.text.strip()
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    ids = json.loads(match.group(0) if match else raw)

    # The model occasionally returns an ID that doesn't match any candidate
    # (truncated/hallucinated) — silently dropping those (instead of trusting
    # them) means Stage 9's chunks_by_id.loc[...] can never KeyError on a
    # selection this function produced. Falls back to all candidates if that
    # leaves nothing, same as the already-empty-response case below.
    valid_ids = set(candidates_df["chunk_id"])
    ids = [i for i in ids if i in valid_ids]

    if not ids:
        ids = candidates_df["chunk_id"].tolist()
    return ids, resp.usage_metadata

In [131]:
def run_selection_all(df, index_dir, rerank_path, out_path, generation_model, sync_every=None, sync_fn=None):
    """Stage 8: filter Stage 7's reranked chunks down to what's actually needed."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "selection_errors.log"
    rerank_records = _load_stage_records(rerank_path)
    completed = _load_completed_ids(out_path)
    synced_through = 0
    doc_cache = {}
    results = {"done": [], "skipped": [], "failed": [], "missing_rerank": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if fb_id not in rerank_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — no rerank output yet")
            results["missing_rerank"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: selecting ...")
        t0 = time.time()
        try:
            rr = rerank_records[fb_id]
            if row.doc_name not in doc_cache:
                chunks_df, _ = load_index(index_dir, row.doc_name)
                doc_cache[row.doc_name] = chunks_df.set_index("chunk_id")
            chunks_by_id = doc_cache[row.doc_name]
            candidates_df = chunks_by_id.loc[rr["chunk_ids"]].reset_index()

            selected_ids, usage = select_chunks(genai_client, row.question, candidates_df, generation_model)
            cost_tracker.log(pipeline="vector_rag", stage="selection", model=generation_model,
                              input_tokens=usage.prompt_token_count, output_tokens=usage.candidates_token_count,
                              doc_name=row.doc_name, financebench_id=fb_id)

            kept_pages = candidates_df.loc[candidates_df.chunk_id.isin(selected_ids), "page_num"].tolist()
            print(f"    kept pages: {kept_pages}  ({len(selected_ids)}/{len(candidates_df)} chunks)")
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "selected_chunk_ids": selected_ids, "n_candidates": len(candidates_df),
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            print(f"    done — kept {len(selected_ids)}/{len(candidates_df)} chunks")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

        # Same Gemini free-tier RPM cap Stage 5 already paces for — Stage 8
        # calls the same model, so it needs the same proactive spacing instead
        # of relying on with_retry's reactive 429 backoff alone.
        time.sleep(max(0.0, MIN_SECONDS_BETWEEN_CALLS - (time.time() - t0)))

        if sync_fn is not None and sync_every and len(results["done"]) - synced_through >= sync_every:
            print(f"    -- syncing after {len(results['done'])} newly selected questions --")
            sync_fn()
            synced_through = len(results["done"])

    if sync_fn is not None and len(results["done"]) > synced_through:
        sync_fn()

    print(f"\nSelection summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_rerank'])} missing rerank output")
    if results["failed"]:
        print(f"Failed questions (see {errors_log}): {results['failed']}")
    return results

In [132]:
selection_results = run_selection_all(
    df=df,
    index_dir=INDEX_DIR,
    rerank_path=RERANK_PATH,
    out_path=SELECTION_PATH,
    generation_model=GENERATION_MODEL,
    sync_every=15,
    sync_fn=sync_selection_to_github,
)

[1/150] financebench_id_03029: selecting ...
    kept pages: [59]  (1/10 chunks)
    done — kept 1/10 chunks
[2/150] financebench_id_04672: selecting ...
    kept pages: [57]  (1/10 chunks)
    done — kept 1/10 chunks
[3/150] financebench_id_00499: selecting ...
    kept pages: [33, 38, 123]  (3/10 chunks)
    done — kept 3/10 chunks
[4/150] financebench_id_01226: selecting ...
    kept pages: [26, 20, 19, 26]  (4/10 chunks)
    done — kept 4/10 chunks
[5/150] financebench_id_01865: selecting ...
    kept pages: [31, 32, 29]  (3/10 chunks)
    done — kept 3/10 chunks
[6/150] financebench_id_00807: selecting ...
    kept pages: [4, 70, 4]  (3/10 chunks)
    done — kept 3/10 chunks
[7/150] financebench_id_00941: selecting ...
    kept pages: [0]  (1/10 chunks)
    done — kept 1/10 chunks
[8/150] financebench_id_01858: selecting ...
    transient error (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman), retrying once ...
    kept pages: 

### Sync selection results back to GitHub

Auto-syncs every 15 questions above; kept here as a manual one-off.

In [133]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_stage_selection.jsonl experiments/results/vector_rag_costs.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync selection stage" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
(nothing new to commit)
Everything up-to-date


---
## Stage 9 — Generate the answer

Answers using only Stage 8's selected chunks as context. **Resumable** — needs
Stage 8 to have produced a selection record for the question first.

In [134]:
GENERATION_PROMPT = """You are a financial analyst answering a question using only the excerpts \
below from a company's SEC filing. Answer concisely and precisely, matching the format the question \
expects (a number, a yes/no with brief reasoning, etc). If the excerpts don't contain enough \
information to answer, say so explicitly rather than guessing.

Question: {question}

Excerpts:
{passages}

Answer:"""


def generate_answer(genai_client, question, context_df, model):
    resp = _generate(genai_client, model, GENERATION_PROMPT.format(question=question, passages=format_passages(context_df)))
    return resp.text.strip(), resp.usage_metadata

In [135]:
def run_generation_all(df, index_dir, selection_path, out_path, generation_model, sync_every=None, sync_fn=None):
    """Stage 9: generate the final answer from Stage 8's selected chunks."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "generation_errors.log"
    selection_records = _load_stage_records(selection_path)
    completed = _load_completed_ids(out_path)
    synced_through = 0
    doc_cache = {}
    results = {"done": [], "skipped": [], "failed": [], "missing_selection": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if fb_id not in selection_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — no selection output yet")
            results["missing_selection"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: generating ...")
        t0 = time.time()
        try:
            sel = selection_records[fb_id]
            if row.doc_name not in doc_cache:
                chunks_df, _ = load_index(index_dir, row.doc_name)
                doc_cache[row.doc_name] = chunks_df.set_index("chunk_id")
            chunks_by_id = doc_cache[row.doc_name]
            selected_df = chunks_by_id.loc[sel["selected_chunk_ids"]].reset_index()

            answer, usage = generate_answer(genai_client, row.question, selected_df, generation_model)
            cost_tracker.log(pipeline="vector_rag", stage="generation", model=generation_model,
                              input_tokens=usage.prompt_token_count, output_tokens=usage.candidates_token_count,
                              doc_name=row.doc_name, financebench_id=fb_id)

            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name, "question": row.question,
                "gold_answer": row.answer, "model_answer": answer,
                "generation_model": generation_model, "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            print(f"    gold : {row.answer[:90]!r}")
            print(f"    model: {answer[:90]!r}")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

        # Same Gemini free-tier RPM cap Stage 5 already paces for — Stage 9
        # calls the same model, so it needs the same proactive spacing instead
        # of relying on with_retry's reactive 429 backoff alone.
        time.sleep(max(0.0, MIN_SECONDS_BETWEEN_CALLS - (time.time() - t0)))

        if sync_fn is not None and sync_every and len(results["done"]) - synced_through >= sync_every:
            print(f"    -- syncing after {len(results['done'])} newly generated answers --")
            sync_fn()
            synced_through = len(results["done"])

    if sync_fn is not None and len(results["done"]) > synced_through:
        sync_fn()

    print(f"\nGeneration summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_selection'])} missing selection output")
    if results["failed"]:
        print(f"Failed questions (see {errors_log}): {results['failed']}")
    return results

In [136]:
generation_results = run_generation_all(
    df=df,
    index_dir=INDEX_DIR,
    selection_path=SELECTION_PATH,
    out_path=GENERATION_PATH,
    generation_model=GENERATION_MODEL,
    sync_every=15,
    sync_fn=sync_generation_to_github,
)

[1/150] financebench_id_03029: generating ...
    gold : '$1577.00'
    model: '1,577'
[2/150] financebench_id_04672: generating ...
    gold : '$8.70'
    model: '8.738 USD billions'
[3/150] financebench_id_00499: generating ...
    gold : 'No, the company is managing its CAPEX and Fixed Assets pretty efficiently, which is eviden'
    model: 'The provided excerpts do not contain enough information to categorize 3M as a "capital-int'
[4/150] financebench_id_01226: generating ...
    gold : 'Operating Margin for 3M in FY2022 has decreased by 1.7% primarily due to: \n-Decrease in gr'
    model: "The 1.7% decrease in 3M's operating margin for 2022 (from 20.8% to 19.1%) was driven by th"
[5/150] financebench_id_01865: generating ...
    gold : 'The consumer segment shrunk by 0.9% organically.'
    model: 'The Consumer Business segment. Based on the provided excerpts, when excluding the impact o'
[6/150] financebench_id_00807: generating ...
    gold : "No. The quick ratio for 3M was 0.96 b

### Sync generated answers back to GitHub

Auto-syncs every 15 questions above; kept here as a manual one-off.

In [137]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_stage_generation.jsonl experiments/results/vector_rag_costs.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync generation stage" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
(nothing new to commit)
Everything up-to-date


---
## Stage 10 — Score each answer

Classifies every answer into one of Islam et al.'s three FinanceBench
categories: Correct / Incorrect / Failure to Answer. Deterministic numeric
matching runs first (handles "1.2 billion" vs "1,200 million" and small
rounding via a 1% relative tolerance) and only fires when both the gold answer
and the model answer contain an extractable number; everything else falls
through to the LLM judge (`openai/gpt-oss-120b` via Groq's free tier — a
different model family from both generation models, which is what matters for
avoiding self-enhancement bias). Spot-check 10-15% of judge outputs by hand
before trusting them for the real experiment.

**Resumable** — needs Stage 9 to have produced a generated answer for the
question first.

In [138]:
from typing import Literal

Label = Literal["Correct", "Incorrect", "Failure to Answer"]

_MAGNITUDE_WORDS = {
    "trillion": 1e12, "tn": 1e12, "billion": 1e9, "bn": 1e9,
    "million": 1e6, "mm": 1e6, "thousand": 1e3,
}

# (?<![A-Za-z0-9]) / (?![A-Za-z0-9]) guard against matching digits glued to
# letters — without them "3M" yields a spurious "3" and "FY2018" yields a
# spurious "2018". A guard on just the single preceding/following character
# (excluding letters only) isn't enough: finditer can still start a fresh match
# *inside* a blocked digit run (e.g. the "0" in "FY2018" is preceded by "2", a
# digit not a letter), so excluding digits too forces the whole contiguous run
# to match-or-skip as one.
#
# The two num alternatives matter: `\d{1,3}(?:,\d{3})+` only matches numbers
# that actually contain comma grouping; `\d+` handles plain digit runs with no
# commas. An earlier version used `(?:,\d{3})*` (zero or more), which let the
# first alternative match just the first 1-3 digits of a longer comma-less
# number and silently truncate it (e.g. "1577.00" -> "157").
_NUM_RE = re.compile(
    r"(?<![A-Za-z0-9])"
    r"(?P<num>-?\$?\s*(?:\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?))"
    r"\s*(?P<mag>trillion|billion|million|thousand|bn|mm|tn|%)?"
    r"(?![A-Za-z0-9])",
    re.IGNORECASE,
)

REL_TOLERANCE = 0.01  # 1% — covers "small rounding" per the project brief
ABS_TOLERANCE = 0.005  # for gold values near zero, where relative tolerance breaks down

_QUESTION_UNIT_RE = re.compile(r"in\s+(?:usd\s+)?(thousand|million|billion|trillion)s?\b", re.IGNORECASE)


def infer_question_unit_multiplier(question: str) -> float:
    """FinanceBench gold answers are bare numbers already expressed in whatever
    unit the question asks for (e.g. "(in USD millions)"), with no unit word
    repeated in the answer itself. A model's free-text answer usually spells
    the unit back out ("$1,577 million"). This scans the question for that unit
    phrase so both sides normalize to the same base scale."""
    m = _QUESTION_UNIT_RE.search(question)
    return _MAGNITUDE_WORDS[m.group(1).lower()] if m else 1.0


def extract_numbers(text: str, default_multiplier: float = 1.0) -> list[float]:
    values = []
    for m in _NUM_RE.finditer(text):
        raw = m.group("num").replace("$", "").replace(",", "").strip()
        if not raw or raw in ("-", "."):
            continue
        try:
            value = float(raw)
        except ValueError:
            continue
        mag = m.group("mag")
        if mag == "%":
            multiplier = 1.0
        elif mag:
            multiplier = _MAGNITUDE_WORDS[mag.lower()]
        else:
            multiplier = default_multiplier
        values.append(value * multiplier)
    return values


def numbers_match(a: float, b: float) -> bool:
    if abs(a - b) <= ABS_TOLERANCE:
        return True
    denom = max(abs(a), abs(b))
    return denom > 0 and abs(a - b) / denom <= REL_TOLERANCE


@dataclass
class ScoreResult:
    label: str
    method: str
    gold_value: float | None = None
    matched_value: float | None = None
    reasoning: str | None = None


def score_deterministic(question, gold_answer, model_answer):
    """Returns a ScoreResult if both answers yield extractable numbers, else None (defer to judge)."""
    unit_multiplier = infer_question_unit_multiplier(question)
    gold_numbers = extract_numbers(gold_answer, default_multiplier=unit_multiplier)
    model_numbers = extract_numbers(model_answer, default_multiplier=unit_multiplier)
    if not gold_numbers or not model_numbers:
        return None

    gold_value = gold_numbers[0]  # FinanceBench gold answers are short; first number is the answer
    for candidate in model_numbers:
        if numbers_match(gold_value, candidate):
            return ScoreResult(label="Correct", method="deterministic", gold_value=gold_value, matched_value=candidate)
    return ScoreResult(label="Incorrect", method="deterministic", gold_value=gold_value, matched_value=model_numbers[0])

In [139]:
JUDGE_PROMPT = """You are grading an AI system's answer to a question about a company's SEC filing, \
against a human-verified gold answer. Classify the model's answer into exactly one of three categories:

- "Correct": the model's answer matches the gold answer in substance (numbers may be phrased \
differently — e.g. rounding, units — but must be materially the same value or conclusion).
- "Incorrect": the model gave a definite answer, but it's wrong.
- "Failure to Answer": the model declined to answer, said the information wasn't available/sufficient, \
or gave a non-answer, rather than committing to a (possibly wrong) answer.

Question: {question}

Gold answer: {gold_answer}

Model answer: {model_answer}

Respond with ONLY a JSON object, no other text: {{"label": "<one of the three categories above>", "reasoning": "<one sentence>"}}"""


def score_with_judge(question, gold_answer, model_answer, client, model="openai/gpt-oss-120b"):
    """client is a groq.Groq instance (OpenAI-compatible chat completions API).
    Returns (ScoreResult, usage) where usage has input/output token counts."""
    resp = client.chat.completions.create(
        model=model, max_tokens=300,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(question=question, gold_answer=gold_answer, model_answer=model_answer)}],
    )
    raw = resp.choices[0].message.content.strip()
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    parsed = json.loads(match.group(0) if match else raw)

    label = parsed["label"]
    if label not in ("Correct", "Incorrect", "Failure to Answer"):
        raise ValueError(f"Judge returned an unrecognized label: {label!r}")

    result = ScoreResult(label=label, method="llm_judge", reasoning=parsed.get("reasoning"))
    usage = {"input_tokens": resp.usage.prompt_tokens, "output_tokens": resp.usage.completion_tokens}
    return result, usage


def score_answer(question, gold_answer, model_answer, judge_client=None, judge_model="openai/gpt-oss-120b"):
    """Deterministic matching first; falls back to the LLM judge if numbers
    aren't extractable from both answers. Returns (ScoreResult, usage | None) —
    usage is only populated when the judge was actually called."""
    result = score_deterministic(question, gold_answer, model_answer)
    if result is not None:
        return result, None

    if judge_client is None:
        raise ValueError(
            "Deterministic matching failed (no extractable number in gold and/or model answer) "
            "and no judge_client was provided to fall back to the LLM judge."
        )
    return score_with_judge(question, gold_answer, model_answer, judge_client, judge_model)


_score_answer = with_retry()(score_answer)

In [140]:
def run_scoring_all(df, generation_path, out_path, judge_client, judge_model, sync_every=None, sync_fn=None):
    """Stage 10: score each Stage 9 answer against the gold answer."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "scoring_errors.log"
    generation_records = _load_stage_records(generation_path)
    completed = _load_completed_ids(out_path)
    synced_through = 0
    results = {"done": [], "skipped": [], "failed": [], "missing_generation": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if fb_id not in generation_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — no generated answer yet")
            results["missing_generation"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: scoring ...")
        try:
            gen = generation_records[fb_id]
            score_result, usage = _score_answer(row.question, row.answer, gen["model_answer"],
                                                  judge_client=judge_client, judge_model=judge_model)
            if usage is not None:
                cost_tracker.log(pipeline="vector_rag", stage="judge", model=judge_model,
                                  input_tokens=usage["input_tokens"], output_tokens=usage["output_tokens"],
                                  doc_name=row.doc_name, financebench_id=fb_id)

            if score_result.reasoning:
                print(f"    reasoning: {score_result.reasoning[:110]!r}")
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "score_label": score_result.label, "score_method": score_result.method,
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            print(f"    done — {score_result.label} ({score_result.method})")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

        if sync_fn is not None and sync_every and len(results["done"]) - synced_through >= sync_every:
            print(f"    -- syncing after {len(results['done'])} newly scored questions --")
            sync_fn()
            synced_through = len(results["done"])

    if sync_fn is not None and len(results["done"]) > synced_through:
        sync_fn()

    print(f"\nScoring summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_generation'])} missing generated answer")
    if results["failed"]:
        print(f"Failed questions (see {errors_log}): {results['failed']}")
    return results

In [141]:
scoring_results = run_scoring_all(
    df=df,
    generation_path=GENERATION_PATH,
    out_path=SCORING_PATH,
    judge_client=judge_client,
    judge_model=JUDGE_MODEL,
    sync_every=15,
    sync_fn=sync_scoring_to_github,
)

[1/150] financebench_id_03029: scoring ...
    done — Correct (deterministic)
[2/150] financebench_id_04672: scoring ...
    done — Correct (deterministic)
[3/150] financebench_id_00499: scoring ...
    done — Incorrect (deterministic)
[4/150] financebench_id_01226: scoring ...
    done — Correct (deterministic)
[5/150] financebench_id_01865: scoring ...
    done — Incorrect (deterministic)
[6/150] financebench_id_00807: scoring ...
    done — Incorrect (deterministic)
[7/150] financebench_id_00941: scoring ...
    done — Incorrect (deterministic)
[8/150] financebench_id_01858: scoring ...
    done — Correct (deterministic)
[9/150] financebench_id_02987: scoring ...
    done — Correct (deterministic)
[10/150] financebench_id_07966: scoring ...
    done — Incorrect (deterministic)
[11/150] financebench_id_04735: scoring ...
    reasoning: 'The model claimed insufficient information and did not provide the numeric ratio, whereas a definitive answer '
    done — Failure to Answer (llm_jud

### Sync scores back to GitHub

Auto-syncs every 15 questions above; kept here as a manual one-off.

In [142]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_stage_scoring.jsonl experiments/results/vector_rag_costs.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync scoring stage" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit
!git -C "{REPO_ROOT}" push origin main

fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
fatal: Unable to create '/content/drive/MyDrive/financebench_project/.git/index.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
(nothing new to commit)
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Already up to date.
Everything up-to-date


---
## Stage 11 — Summarize

Answer-quality breakdown, retrieval Recall@k/MRR@k (hybrid vs. reranked), and
token totals per stage, computed over whatever's in Stages 6-10's output files
so far — doesn't require all 150 to be done. No latency number here — see the
trade-off note at the top of this notebook.

In [143]:
import statistics
from collections import Counter, defaultdict


def summarize_results(hybrid_path, rerank_path, scoring_path, cost_log_path=None):
    hybrid = _load_stage_records(hybrid_path)
    rerank = _load_stage_records(rerank_path)
    scoring = _load_jsonl(scoring_path)
    if not scoring:
        print(f"No scored results yet in {scoring_path}")
        return {}

    n = len(scoring)
    label_counts = Counter(r["score_label"] for r in scoring)

    def _mean_metric(records_by_id, field, k):
        vals = [r[field][str(k)] for r in records_by_id.values()]
        return sum(vals) / len(vals) if vals else float("nan")

    retrieval_summary = {
        "hybrid": {
            "recall_at_k": {k: _mean_metric(hybrid, "recall_at_k", k) for k in K_VALUES},
            "mrr_at_k": {k: _mean_metric(hybrid, "mrr_at_k", k) for k in K_VALUES},
        },
        "rerank": {
            "recall_at_k": {k: _mean_metric(rerank, "recall_at_k", k) for k in K_VALUES},
            "mrr_at_k": {k: _mean_metric(rerank, "mrr_at_k", k) for k in K_VALUES},
        },
    }

    summary = {
        "n_questions": n,
        "answer_quality": {label: {"count": c, "pct": round(100 * c / n, 1)} for label, c in label_counts.items()},
        "retrieval": retrieval_summary,
    }

    if cost_log_path is not None:
        cost_records = _load_jsonl(cost_log_path)
        by_stage = defaultdict(lambda: {"input_tokens": 0, "output_tokens": 0, "calls": 0, "cost_usd": 0.0, "cost_unpriced_calls": 0})
        for r in cost_records:
            s = by_stage[r["stage"]]
            s["input_tokens"] += r["input_tokens"]
            s["output_tokens"] += r["output_tokens"]
            s["calls"] += 1
            if r["cost_usd"] is not None:
                s["cost_usd"] += r["cost_usd"]
            else:
                s["cost_unpriced_calls"] += 1
        summary["tokens_by_stage"] = dict(by_stage)
        total_cost = sum(s["cost_usd"] for s in by_stage.values())
        total_unpriced = sum(s["cost_unpriced_calls"] for s in by_stage.values())
        summary["cost"] = {
            "total_usd": total_cost, "usd_per_question": total_cost / n if n else float("nan"),
            "unpriced_calls": total_unpriced,
        }

    return summary


def print_summary(summary: dict) -> None:
    if not summary:
        return
    print(f"Questions scored: {summary['n_questions']}")
    print("\nAnswer quality:")
    for label, stats in summary["answer_quality"].items():
        print(f"  {label:<20} {stats['count']:>4}  ({stats['pct']}%)")

    print("\nRetrieval quality (mean across questions):")
    for stage, m in summary["retrieval"].items():
        print(f"  {stage}:")
        print(f"    Recall@k: {m['recall_at_k']}")
        print(f"    MRR@k:    {m['mrr_at_k']}")

    if "tokens_by_stage" in summary:
        print("\nTokens by stage (the meaningful cost signal right now — every model in\n              use is free-tier or self-hosted, so token volume is what actually differs\n              between stages/pipelines, not dollars):")
        for stage, t in summary["tokens_by_stage"].items():
            print(f"  {stage:<18} calls={t['calls']:<5} in={t['input_tokens']:<10} out={t['output_tokens']}")
        c = summary["cost"]
        if c["unpriced_calls"]:
            print(f"\nDollar cost: ${c['total_usd']:.4f} total (${c['usd_per_question']:.5f}/question) — "
                  f"NOT the real total: {c['unpriced_calls']} calls have no confirmed price yet "
                  f"(e.g. Jina rerank, DeepSeek V4 — see PRICING_PER_MILLION_TOKENS) and are excluded, "
                  f"not counted as $0. Fill in their prices before quoting this number in the thesis.")
        else:
            print(f"\nDollar cost: ${c['total_usd']:.4f} total (${c['usd_per_question']:.5f}/question) — "
                  f"genuinely $0 across the board (Gemini/Groq free tiers, Stella self-hosted); "
                  f"becomes meaningful once DeepSeek V4 (not free) is wired in.")

    print("\nNote: per-question end-to-end latency is not measured by Stages 6-10 — "
          "they're independently resumable and can run hours/days apart. Measure it "
          "separately via a dedicated sequential timing pass before it goes in the thesis.")

In [144]:
summary = summarize_results(HYBRID_PATH, RERANK_PATH, SCORING_PATH, cost_log_path=COSTS_PATH)
print_summary(summary)

Questions scored: 147

Answer quality:
  Correct                88  (59.9%)
  Incorrect              46  (31.3%)
  Failure to Answer      13  (8.8%)

Retrieval quality (mean across questions):
  hybrid:
    Recall@k: {5: 0.7466666666666667, 10: 0.8322222222222222, 15: 0.8566666666666667}
    MRR@k:    {5: 0.5825555555555556, 10: 0.5934126984126984, 15: 0.5951298701298702}
  rerank:
    Recall@k: {5: nan, 10: nan, 15: nan}
    MRR@k:    {5: nan, 10: nan, 15: nan}

Tokens by stage (the meaningful cost signal right now — every model in
              use is free-tier or self-hosted, so token volume is what actually differs
              between stages/pipelines, not dollars):
  query_expansion    calls=461   in=54514      out=20898
  rerank             calls=481   in=2831954    out=0
  selection          calls=476   in=2558471    out=18045
  generation         calls=498   in=524311     out=29491
  judge              calls=113   in=36496      out=14814

Dollar cost: $0.0000 total ($0.00000/

### Next steps
1. Check `INDEX_DIR/indexing_errors.log`, `QUERIES_DIR/embedding_errors.log`, and
   the `*_errors.log` files next to each Stage 6-10 output for anything that
   failed and needs a second look.
2. Fill in `PRICING_PER_MILLION_TOKENS` in Stage 1 with current published
   rates — token counts are logged throughout, but `cost_usd` stays `None`
   until those are filled in.
3. Once this is stable, wire in DeepSeek V4 as a second `generation_model` and
   re-run Stages 8-10 with it (Stages 4-7's indexes, query embeddings, and
   retrieval/rerank results are model-independent and don't need to be redone).
4. Run a dedicated sequential timing pass (all steps back-to-back, no
   resuming) over some/all questions for the median-of-N latency number the
   project brief asks for — Stages 6-10 above don't produce one.

---
## Stage 12 — Dedicated latency timing pass

Stages 6–10 are each independently resumable, so two stages for the same
question can run hours or days apart — that's exactly why they can't produce
a real "query submission to answer" latency number (see the trade-off note
at the top of this notebook). This stage is different on purpose: a small,
**non-resumable-per-run, strictly sequential** pass over a fixed sample of
questions, each one run **fully fresh** (no cached query embedding, no
cached rerank/selection state) straight through query expansion → embed →
hybrid retrieve → rerank → select → generate, timed end-to-end and per
sub-stage. Scoring (Stage 10) is excluded — grading an answer isn't part of
the system's response time.

Document indexing (Stage 4) is excluded too: that's one-time preprocessing
per document, not something a live query pays for, so a question can only be
sampled here if its document is already indexed.

Per the project brief: run each question multiple times, report the median.
Default here is **10 questions × 3 repeats = 30 timed passes**, with a
40s gap between passes — a value inherited from clearing Voyage's 3 RPM
window (see Stage 7); not yet re-verified against Jina's actual rate limits,
so tighten it once real 429 behavior against Jina is observed.

In [145]:
LATENCY_SAMPLE_SIZE = 10   # questions
LATENCY_REPEATS = 3        # per project brief: run each query multiple times, report median
LATENCY_GAP_SECONDS = 40   # between timed passes -- inherited from Voyage's 3 RPM window (see Stage 7);
                            # not yet re-verified against Jina's actual rate limits
LATENCY_PATH = RESULTS_DIR / "vector_rag_latency.jsonl"


def sync_latency_to_github():
    _git_sync(["experiments/results/vector_rag_latency.jsonl"], "Sync latency timing pass")


def time_single_pass(row, chunks_df, dense_embeddings, bm25_index, embed_model, jina_api_key,
                      genai_client, generation_model, top_k_hybrid=10, top_k_rerank=10):
    """One fully fresh, uncached pass through query expansion -> embed ->
    hybrid retrieve -> rerank -> select -> generate, for a single question.
    Returns (timings_sec dict, model_answer). The document's index/BM25 are
    the caller's job to load once, outside the timed loop -- that's
    preprocessing a live system would keep warm in memory, not per-query work.
    top_k defaults match Stages 6/7's current config (10/10, see the funnel-
    size note); keep these in sync if that changes."""
    timings = {}
    t_total0 = time.time()

    t0 = time.time()
    expanded_query, _ = expand_query(genai_client, row.question, generation_model)
    timings["query_expansion"] = time.time() - t0

    t0 = time.time()
    query_vec = embed_model.encode(format_query(expanded_query), normalize_embeddings=True)
    timings["query_embedding"] = time.time() - t0

    t0 = time.time()
    top_hybrid = hybrid_retrieve(chunks_df, dense_embeddings, bm25_index, query_vec, expanded_query, top_k=top_k_hybrid)
    timings["hybrid_retrieval"] = time.time() - t0

    t0 = time.time()
    top_rerank, _ = rerank(jina_api_key, expanded_query, top_hybrid, top_k=top_k_rerank)
    timings["rerank"] = time.time() - t0

    t0 = time.time()
    selected_ids, _ = select_chunks(genai_client, row.question, top_rerank, generation_model)
    selected_df = top_rerank[top_rerank["chunk_id"].isin(selected_ids)]
    timings["selection"] = time.time() - t0

    t0 = time.time()
    answer, _ = generate_answer(genai_client, row.question, selected_df, generation_model)
    timings["generation"] = time.time() - t0

    timings["total"] = time.time() - t_total0
    return timings, answer

In [146]:
def run_latency_pass(df, index_dir, embed_model, jina_api_key, genai_client, generation_model, out_path,
                      sample_size=LATENCY_SAMPLE_SIZE, repeats=LATENCY_REPEATS, gap_seconds=LATENCY_GAP_SECONDS,
                      seed=42, sync_fn=None):
    """Runs the dedicated latency sample. Each (question, repeat) pair is
    timed as one uninterrupted sequential pass (see time_single_pass) -- but
    the pass *list* itself is resumable the normal way, so a disconnect
    partway through a 30-run sample doesn't mean starting over: already-timed
    (financebench_id, repeat) pairs already in out_path are skipped."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    indexed = df[df.doc_name.apply(lambda d: is_indexed(index_dir, d))]
    if len(indexed) < len(df):
        print(f"Note: {len(df) - len(indexed)} questions skipped from the sampling pool -- their document isn't indexed yet")
    sample = indexed.sample(n=min(sample_size, len(indexed)), random_state=seed)

    completed_pairs = {(r["financebench_id"], r["repeat"]) for r in _load_jsonl(out_path)}
    total_runs = len(sample) * repeats
    print(f"Timing {len(sample)} questions x {repeats} repeats = {total_runs} passes "
          f"({len(completed_pairs)} already done) -- >= {gap_seconds * (total_runs - len(completed_pairs)) / 60:.0f} "
          f"min remaining just from pacing, plus actual work time")

    doc_cache = {}
    for qi, row in enumerate(sample.itertuples(), start=1):
        if row.doc_name not in doc_cache:
            chunks_df, dense_embeddings = load_index(index_dir, row.doc_name)
            doc_cache[row.doc_name] = (chunks_df, dense_embeddings, build_bm25(chunks_df))
        chunks_df, dense_embeddings, bm25_index = doc_cache[row.doc_name]

        for r in range(1, repeats + 1):
            if (row.financebench_id, r) in completed_pairs:
                continue
            print(f"[{qi}/{len(sample)} x {r}/{repeats}] {row.financebench_id} ({row.doc_name}) ...")
            try:
                timings, answer = time_single_pass(row, chunks_df, dense_embeddings, bm25_index,
                                                     embed_model, jina_api_key, genai_client, generation_model)
                record = {
                    "financebench_id": row.financebench_id, "doc_name": row.doc_name,
                    "repeat": r, "timings_sec": timings, "timestamp": time.time(),
                }
                with out_path.open("a") as f:
                    f.write(json.dumps(record) + "\n")
                print(f"    total: {timings['total']:.2f}s  "
                      f"(expand={timings['query_expansion']:.2f} embed={timings['query_embedding']:.2f} "
                      f"hybrid={timings['hybrid_retrieval']:.2f} rerank={timings['rerank']:.2f} "
                      f"select={timings['selection']:.2f} generate={timings['generation']:.2f})")
            except Exception as e:
                print(f"    FAILED: {e}")
            time.sleep(gap_seconds)

    if sync_fn is not None:
        sync_fn()

In [ ]:
latency_results = run_latency_pass(
    df=df,
    index_dir=INDEX_DIR,
    embed_model=embed_model,
    jina_api_key=JINA_API_KEY,
    genai_client=genai_client,
    generation_model=GENERATION_MODEL,
    out_path=LATENCY_PATH,
    sync_fn=sync_latency_to_github,
)

Timing 10 questions x 3 repeats = 30 passes (0 already done) -- >= 20 min remaining just from pacing, plus actual work time
[1/10 x 1/3] financebench_id_00005 (CORNING_2022_10K) ...


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


    total: 11.12s  (expand=2.72 embed=0.03 hybrid=0.00 rerank=0.44 select=4.27 generate=3.65)
[1/10 x 2/3] financebench_id_00005 (CORNING_2022_10K) ...


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


    total: 8.10s  (expand=2.56 embed=0.04 hybrid=0.00 rerank=0.45 select=4.21 generate=0.84)
[1/10 x 3/3] financebench_id_00005 (CORNING_2022_10K) ...


...

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


    total: 9.11s  (expand=1.69 embed=0.03 hybrid=0.00 rerank=0.47 select=2.55 generate=4.36)
[2/10 x 3/3] financebench_id_06655 (AMAZON_2017_10K) ...


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


    transient error (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman), retrying once ...
    rate limited (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please ), backing off 5s ...
    rate limited (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please ), backing off 10s ...
    rate limited (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please ), backing off 20s ...
    rate limited (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please ), backing off 40s ...
    rate limited (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please ), backing off 80s ...
    rate limited (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please ), backing off 120s ...
    FAILED: 

: 

### Sync latency results back to GitHub

Already synced once at the end of the run above; kept here as a manual one-off.

In [ ]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_latency.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync latency timing pass" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit
!git -C "{REPO_ROOT}" push origin main

The following paths are ignored by one of your .gitignore files:
experiments/results/vector_rag_latency.jsonl
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    experiments/results/vector_rag_results.jsonl

no changes added to commit (use "git add" and/or "git commit -a")
(nothing new to commit)
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Already up to date.
Everything up-to-date


In [ ]:
def summarize_latency(out_path):
    """Median-of-N per question (per the project brief), then median across
    questions for the headline number -- robust to one slow or retried pass
    dragging the number the way a mean would."""
    records = _load_jsonl(out_path)
    if not records:
        print(f"No latency data yet in {out_path}")
        return {}

    by_question = defaultdict(list)
    for r in records:
        by_question[r["financebench_id"]].append(r["timings_sec"]["total"])
    per_question_median = {fb_id: statistics.median(vals) for fb_id, vals in by_question.items()}
    overall_median = statistics.median(per_question_median.values())

    stage_names = [k for k in records[0]["timings_sec"] if k != "total"]
    stage_medians = {stage: statistics.median(r["timings_sec"][stage] for r in records) for stage in stage_names}

    return {
        "n_questions": len(by_question), "n_runs": len(records),
        "per_question_median_sec": per_question_median,
        "overall_median_sec": overall_median,
        "stage_medians_sec": stage_medians,
    }


def print_latency_summary(summary: dict) -> None:
    if not summary:
        return
    print(f"Latency sample: {summary['n_questions']} questions, {summary['n_runs']} total timed passes")
    print(f"\nMedian end-to-end latency (median-of-N per question, then median across questions): "
          f"{summary['overall_median_sec']:.2f}s")
    print("\nMedian latency by stage (across all timed passes):")
    for stage, m in summary["stage_medians_sec"].items():
        print(f"  {stage:<18} {m:.2f}s")


latency_summary = summarize_latency(LATENCY_PATH)
print_latency_summary(latency_summary)

Latency sample: 10 questions, 29 total timed passes

Median end-to-end latency (median-of-N per question, then median across questions): 9.19s

Median latency by stage (across all timed passes):
  query_expansion    2.55s
  query_embedding    0.04s
  hybrid_retrieval   0.00s
  rerank             0.12s
  selection          2.73s
  generation         2.30s


---
## Walkthrough — one example, every function, bottom-up

Picks a single question and runs it through every user-defined function in
this notebook, one at a time, printing each function's actual output before
feeding it into the next -- leaf helpers first, then the function that
composes them (e.g. `sent_tokenize` before `chunk_page` before
`parse_and_chunk_document`), matching the real call tree:

```
index_all_documents        (loops docs, resumable, error-tolerant)
  └─ is_indexed             (skip check)
  └─ index_document          (per-doc work)
       ├─ parse_and_chunk_document
       │    └─ chunk_page              (per-page)
       │         ├─ sent_tokenize
       │         └─ split_oversized_sentence   (rare, table blocks)
       │              └─ _greedy_token_split
       └─ embed_model.encode()
```

...and the same shape repeats down through query expansion, hybrid
retrieval, rerank, selection, generation, and scoring below.

**What this does and doesn't touch:**
- Retrieval/rerank/selection/generation reuse this document's *already-indexed*
  chunks/embeddings via `load_index(...)` -- same chunk IDs as the real
  pipeline, so results are directly comparable to what's already in
  `HYBRID_PATH`/`RERANK_PATH`/etc.
- Every LLM/rerank call below is a **live API call** (Gemini, Jina, Groq),
  not replayed from a saved file -- cheap on the free tiers, but not
  instant, and text output can vary slightly run to run.
- None of these calls go through `cost_tracker.log(...)` -- this is a
  read-only demo, not a pipeline run, so it doesn't add entries to the real
  `vector_rag_costs.jsonl`.
- The resumable batch drivers (`index_all_documents`, `embed_all_queries`,
  `run_hybrid_retrieval_all`, `run_rerank_all`, `run_selection_all`,
  `run_generation_all`, `run_scoring_all`) aren't re-run here -- each is just
  "loop the function below over all 150 questions, skip what's done,
  retry-tolerant." Trivial path helpers (`index_paths`, `query_paths`,
  `is_indexed`, `is_query_embedded`) and the `sync_*_to_github` git helpers
  are skipped too -- nothing to see output-wise beyond a bool or a tuple of
  paths.

In [ ]:
# The example this whole walkthrough follows.
example_row = df[df.financebench_id == "financebench_id_03029"].iloc[0]
demo_doc_name = example_row.doc_name
demo_pdf_path = PDF_DIR / f"{demo_doc_name}.pdf"
demo_gold_pages = [e["evidence_page_num"] for e in example_row.evidence]

print(f"financebench_id: {example_row.financebench_id}")
print(f"doc_name:        {demo_doc_name}")
print(f"question:        {example_row.question}")
print(f"gold answer:     {example_row.answer}")
print(f"gold pages:      {demo_gold_pages}")

### Stage 4 — Parse & chunk (bottom-up)

`sent_tokenize` → `split_oversized_sentence` (rare) → `chunk_page` (one
page) → `parse_and_chunk_document` (whole doc) → `embed_model.encode`
(what `index_document` does internally, minus the disk write).

In [ ]:
# One real page's raw markdown text, picked so it has enough content to be
# a useful demo (skips likely-blank cover/TOC pages).
demo_pages_raw = pymupdf4llm.to_markdown(str(demo_pdf_path), page_chunks=True)
demo_page_dict = next(p for p in demo_pages_raw if len(p["text"].strip()) > 800)
demo_page_num = demo_page_dict["metadata"]["page_number"] - 1  # 0-indexed, matches FinanceBench
demo_page_text = demo_page_dict["text"]

print(f"picked page {demo_page_num} ({len(demo_page_text)} chars)\n")
print(demo_page_text[:600])

In [ ]:
# sent_tokenize -- leaf: splits page text into sentence-ish units.
demo_sentences = sent_tokenize(demo_page_text)
print(f"{len(demo_sentences)} sentences\n")
for s in demo_sentences[:5]:
    print(f"  [{count_tokens(s):>3} tok] {s[:100]!r}")

In [ ]:
# split_oversized_sentence / _greedy_token_split -- rare path, only fires on
# a "sentence" that alone exceeds max_tokens (usually a table block with no
# period-then-capital-letter boundary). Real pages rarely trigger it, so this
# uses a synthetic run-on blob (no periods) to force the path and show it.
demo_table_blob = " ".join(f"Item{i} {i * 137} {i * 29.5}" for i in range(300))
print(f"synthetic blob: {count_tokens(demo_table_blob)} tokens, no sentence boundaries\n")

demo_pieces = split_oversized_sentence(demo_table_blob, count_tokens, max_tokens=512)
print(f"split into {len(demo_pieces)} pieces:")
for p in demo_pieces:
    print(f"  [{count_tokens(p):>3} tok] {p[:80]!r} ...")

In [ ]:
# chunk_page -- one page -> <=512-token chunks, with overlap seeded from the
# tail of the previous chunk.
demo_page_chunks = chunk_page(demo_page_text, demo_page_num, demo_doc_name, count_tokens)
print(f"page {demo_page_num} -> {len(demo_page_chunks)} chunk(s)\n")
for c in demo_page_chunks:
    print(f"  chunk_idx={c['chunk_idx']}  {c['token_count']} tok  {c['text'][:100]!r}")

In [ ]:
# parse_and_chunk_document -- the whole PDF, all pages, chunked. This is
# exactly what index_document calls before it embeds -- freshly recomputed
# here (not loaded from disk) so you can see the raw parse+chunk output.
demo_chunks_df = parse_and_chunk_document(demo_pdf_path, demo_doc_name, count_tokens)
print(f"\n{demo_doc_name}: {len(demo_chunks_df)} chunks total\n")
demo_chunks_df[["chunk_id", "page_num", "chunk_idx", "token_count"]].head(8)

In [ ]:
# embed_model.encode -- what index_document does with parse_and_chunk_document's
# output, on a small slice (the real Stage 4 does all of them, batch_size=8).
# Not saved to disk here -- index_document/index_all_documents already did
# that for this document; re-running the full embed pass here would just
# waste GPU time recomputing something identical.
demo_embeddings_preview = embed_model.encode(
    demo_chunks_df["text"].tolist()[:5], normalize_embeddings=True, convert_to_numpy=True,
)
print(f"shape: {demo_embeddings_preview.shape}  (5 chunks x {demo_embeddings_preview.shape[1]}-dim)")
print(f"first vector, first 8 dims: {demo_embeddings_preview[0][:8]}")

### Query side — expand & embed (bottom-up)

`expand_query` (Gemini call) → `format_query` (wraps it for the
instruct-tuned embedder) → `embed_model.encode` (same model, query side —
this is what `embed_all_queries` does per question).

In [ ]:
# expand_query -- rewrites the question into a denser search query before
# embedding (synonyms, exact line-item names). Live Gemini call.
demo_expanded_query, demo_expand_usage = expand_query(genai_client, example_row.question, GENERATION_MODEL)
print(f"question: {example_row.question!r}\n")
print(f"expanded: {demo_expanded_query!r}\n")
print(f"usage: {demo_expand_usage.prompt_token_count} in / {demo_expand_usage.candidates_token_count} out")

In [ ]:
# format_query wraps the expanded text in the instruct-format Stella expects
# for queries (chunks are embedded plain, no wrapper -- see index_document).
demo_formatted_query = format_query(demo_expanded_query)
print(demo_formatted_query)

demo_query_vec = embed_model.encode(demo_formatted_query, normalize_embeddings=True)
print(f"\nquery vector shape: {demo_query_vec.shape}")

### Stage 6 — Hybrid retrieval (bottom-up)

`bm25_tokenize` → `build_bm25` → `hybrid_retrieve` (dense + BM25, α=0.85)
→ `recall_at_k` / `reciprocal_rank_at_k` → `compute_retrieval_metrics`.

Uses `load_index(...)` here rather than the freshly-parsed `demo_chunks_df`
above, so chunk IDs match what's already saved in `HYBRID_PATH` for this
document.

In [ ]:
# bm25_tokenize + build_bm25 -- the sparse side of hybrid retrieval.
demo_indexed_chunks_df, demo_dense_embeddings = load_index(INDEX_DIR, demo_doc_name)
demo_bm25_index = build_bm25(demo_indexed_chunks_df)

demo_bm25_tokens = bm25_tokenize(demo_expanded_query)
print(f"bm25 query tokens: {demo_bm25_tokens}")
print(f"bm25 index built over {len(demo_indexed_chunks_df)} chunks")

In [ ]:
# hybrid_retrieve -- combines dense cosine similarity + normalized BM25 score.
demo_top_hybrid = hybrid_retrieve(
    demo_indexed_chunks_df, demo_dense_embeddings, demo_bm25_index, demo_query_vec, demo_expanded_query, top_k=10,
)
demo_top_hybrid[["chunk_id", "page_num", "hybrid_score", "dense_score", "bm25_score"]]

In [ ]:
# recall_at_k / reciprocal_rank_at_k (leaves) -> compute_retrieval_metrics (composes them).
demo_ranked_pages_hybrid = demo_top_hybrid["page_num"].tolist()

print(f"recall_at_k(top10, gold, k=5)  = {recall_at_k(demo_ranked_pages_hybrid, demo_gold_pages, 5):.3f}")
print(f"reciprocal_rank_at_k(..., k=5) = {reciprocal_rank_at_k(demo_ranked_pages_hybrid, demo_gold_pages, 5):.3f}\n")

demo_hybrid_metrics = compute_retrieval_metrics(
    ranked_pages=demo_ranked_pages_hybrid, gold_pages=demo_gold_pages,
    financebench_id=example_row.financebench_id, doc_name=demo_doc_name, stage="hybrid_top10",
)
demo_hybrid_metrics

### Stage 7 — Rerank (bottom-up)

`_jina_rerank` (raw API call) → `rerank` (budget-trims + wraps it) →
`compute_retrieval_metrics` again, this time on the reranked order.

In [ ]:
# _jina_rerank -- the raw, retry-wrapped Jina rerank API call.
demo_jina_raw = _jina_rerank(
    JINA_API_KEY, demo_expanded_query, demo_top_hybrid["text"].tolist(),
    model="jina-reranker-v2-base-multilingual", top_k=10,
)
print(f"{len(demo_jina_raw['results'])} results, {demo_jina_raw.get('usage', {}).get('total_tokens')} tokens\n")
for r in demo_jina_raw["results"][:3]:
    print(f"  index={r['index']}  relevance_score={r['relevance_score']:.3f}")

In [ ]:
# rerank -- wraps the call above with candidate-budget trimming and returns
# a DataFrame instead of the raw API response.
demo_top_rerank, demo_rerank_tokens = rerank(JINA_API_KEY, demo_expanded_query, demo_top_hybrid, top_k=10)
print(f"rerank tokens (for cost logging): {demo_rerank_tokens}\n")
demo_top_rerank[["chunk_id", "page_num", "rerank_score"]]

In [ ]:
# compute_retrieval_metrics on the reranked order -- compare against the
# hybrid-stage metrics above to see what reranking actually changed.
demo_ranked_pages_rerank = demo_top_rerank["page_num"].tolist()
demo_rerank_metrics = compute_retrieval_metrics(
    ranked_pages=demo_ranked_pages_rerank, gold_pages=demo_gold_pages,
    financebench_id=example_row.financebench_id, doc_name=demo_doc_name, stage="rerank_top10",
)
print(f"hybrid recall@5: {demo_hybrid_metrics.recall_at_k[5]:.3f}   rerank recall@5: {demo_rerank_metrics.recall_at_k[5]:.3f}")
demo_rerank_metrics

### Stage 8 — Selection (bottom-up)

`format_passages` (leaf, also reused by generation below) →
`select_chunks` (Gemini call, filters the 10 reranked chunks down to what's
actually needed).

In [ ]:
# format_passages -- how candidates get serialized into the prompt.
print(format_passages(demo_top_rerank.head(2))[:500])

In [ ]:
# select_chunks -- live Gemini call, drops irrelevant/redundant candidates.
demo_selected_ids, demo_select_usage = select_chunks(genai_client, example_row.question, demo_top_rerank, GENERATION_MODEL)
demo_selected_df = demo_top_rerank[demo_top_rerank.chunk_id.isin(demo_selected_ids)].reset_index(drop=True)

print(f"kept {len(demo_selected_ids)}/{len(demo_top_rerank)} chunks: {demo_selected_ids}")
print(f"kept pages: {demo_selected_df['page_num'].tolist()}")
print(f"usage: {demo_select_usage.prompt_token_count} in / {demo_select_usage.candidates_token_count} out")

### Stage 9 — Generate the answer

`generate_answer` — live Gemini call over Stage 8's selected chunks.

In [ ]:
demo_model_answer, demo_generate_usage = generate_answer(genai_client, example_row.question, demo_selected_df, GENERATION_MODEL)

print(f"question: {example_row.question}\n")
print(f"gold answer:  {example_row.answer}")
print(f"model answer: {demo_model_answer}\n")
print(f"usage: {demo_generate_usage.prompt_token_count} in / {demo_generate_usage.candidates_token_count} out")

### Stage 10 — Score the answer (bottom-up)

`infer_question_unit_multiplier` / `extract_numbers` / `numbers_match`
(leaves) → `score_deterministic` (composes them) → `score_with_judge`
(the LLM-judge fallback, shown regardless of whether deterministic scoring
already succeeded, so you can see it work) → `score_answer` (the real
combined entry point Stage 10 actually calls).

In [ ]:
demo_unit_mult = infer_question_unit_multiplier(example_row.question)
demo_gold_numbers = extract_numbers(example_row.answer, default_multiplier=demo_unit_mult)
demo_model_numbers = extract_numbers(demo_model_answer, default_multiplier=demo_unit_mult)

print(f"unit multiplier inferred from question: {demo_unit_mult}")
print(f"numbers in gold answer:  {demo_gold_numbers}")
print(f"numbers in model answer: {demo_model_numbers}")
if demo_gold_numbers and demo_model_numbers:
    print(f"numbers_match(gold[0], model[0]) = {numbers_match(demo_gold_numbers[0], demo_model_numbers[0])}")

In [ ]:
demo_det_result = score_deterministic(example_row.question, example_row.answer, demo_model_answer)
print(demo_det_result if demo_det_result is not None else "None -- no extractable number on one or both sides, would defer to the judge")

In [ ]:
# score_with_judge -- manual demo of the LLM-judge path regardless of the
# deterministic result above, so you can see it run at least once.
demo_judge_result, demo_judge_usage = score_with_judge(
    example_row.question, example_row.answer, demo_model_answer, judge_client, JUDGE_MODEL,
)
print(demo_judge_result)
print(f"usage: {demo_judge_usage['input_tokens']} in / {demo_judge_usage['output_tokens']} out")

In [ ]:
# score_answer -- the real entry point Stage 10 calls: deterministic first,
# LLM judge only if that fails.
demo_final_result, demo_final_usage = score_answer(
    example_row.question, example_row.answer, demo_model_answer,
    judge_client=judge_client, judge_model=JUDGE_MODEL,
)
print(f"final label: {demo_final_result.label}  (method: {demo_final_result.method})")
demo_final_result

---
**Note found while adding this walkthrough:** the cells after this section
(`build_all_trees` and a few empty ones) are leftover PageIndex/vectorless
code that doesn't belong in this notebook — not touched here, but worth
deleting next time you're in this file.